# Paper Figure Panels

This notebook is a lightweight orchestration layer for paper figures.

Each panel is organized into two stages:

1. Export one or more data bundles with `analyze.py`
2. Plot those bundles with a plotting script

The goal is to keep each panel readable, self-documenting, and easy to duplicate when building the next figure.

## Usage Notes

- Run the setup cell below first so the kernel changes from `notebooks/` to the repository root.
- By default, the execution cells only preview commands.
- Set the relevant `RUN_... = True` toggle when you want to actually launch exports or plotting.
- The command strings are intentionally stored verbatim so this notebook mirrors your shell workflow.
- Use the plot style configuration cell near the top to set shared font and image-output defaults for all plotting stages.
- Group-label convention: use `Intact Control>Kir`, `Intact PFN>Kir`, `AR Control>Kir`, and `AR PFN>Kir` as the base labels; add chamber qualifiers only when needed for disambiguation, formatted like `Intact Control>Kir (flat large)`.

## Setup

Run this cell once after opening or restarting the notebook kernel.

In [ ]:
%cd ..

In [ ]:
from copy import deepcopy
from pathlib import Path
import shlex
import shutil
import subprocess
from IPython.display import Image, Markdown, display

ROOT = Path.cwd()
print(f"Notebook working directory: {ROOT}")

## Plot Style Configuration

Set these once near the top of the notebook, then rerun any plotting cells you want to regenerate.

In [ ]:
PLOT_STYLE = {
    "font_size": 16,
    "font_family": "Arial",
    "image_format": "pdf",
}

display(
    Markdown(
        "\n".join(
            [
                "**Active plot defaults**",
                f"- Font size: `{PLOT_STYLE['font_size']}`",
                f"- Font family: `{PLOT_STYLE['font_family']}`",
                f"- Image format: `.{PLOT_STYLE['image_format'].lstrip('.').lower()}`",
            ]
        )
    )
)


In [ ]:
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".gif", ".pdf", ".svg"}


def normalize_image_format(image_format):
    normalized = image_format.strip().lstrip(".").lower()
    if not normalized:
        raise ValueError("image_format must not be empty")
    return normalized


def replace_image_suffix(path_str, image_format):
    path = Path(path_str)
    if path.suffix.lower() not in IMAGE_EXTENSIONS:
        return path_str
    return str(path.with_suffix(f".{normalize_image_format(image_format)}"))


def set_command_flag(tokens, flag, value):
    value = str(value)
    if flag in tokens:
        index = tokens.index(flag)
        if index + 1 >= len(tokens):
            raise ValueError(f"Flag {flag} is missing its value in command: {shlex.join(tokens)}")
        tokens[index + 1] = value
        return
    tokens.extend([flag, value])


def apply_plot_style_to_command(command, plot_style):
    tokens = shlex.split(command)

    for index in range(len(tokens) - 1):
        if not tokens[index].startswith("--"):
            continue
        tokens[index + 1] = replace_image_suffix(
            tokens[index + 1], plot_style["image_format"]
        )

    set_command_flag(tokens, "--fs", plot_style["font_size"])
    set_command_flag(tokens, "--fontFamily", plot_style["font_family"])
    set_command_flag(tokens, "--imgFormat", normalize_image_format(plot_style["image_format"]))
    return shlex.join(tokens)


def looks_like_plot_command(item):
    return any(key in item for key in ("output_path", "output_paths", "rename_outputs"))


def apply_plot_style_to_item(item, plot_style=None):
    if not looks_like_plot_command(item):
        return deepcopy(item)

    style = plot_style or PLOT_STYLE
    styled_item = deepcopy(item)
    styled_item["command"] = apply_plot_style_to_command(styled_item["command"], style)

    tokens = shlex.split(styled_item["command"])
    for item_key, flag in (
        ("xlabel", "--xlabel"),
        ("ylabel", "--ylabel"),
        ("plot_rewards_xlabel", "--plot-rewards-xlabel"),
        ("plot_rewards_ylabel", "--plot-rewards-ylabel"),
        (
            "btw_rwd_conditioned_disttrav_xlabel",
            "--btw-rwd-conditioned-disttrav-xlabel",
        ),
        (
            "btw_rwd_conditioned_disttrav_ylabel",
            "--btw-rwd-conditioned-disttrav-ylabel",
        ),
        (
            "first_n_reward_diagnostics_xlabel",
            "--first-n-reward-diagnostics-xlabel",
        ),
        (
            "first_n_reward_diagnostics_ylabel",
            "--first-n-reward-diagnostics-ylabel",
        ),
        (
            "turn_prob_dist_xlabel",
            "--turn-prob-dist-xlabel",
        ),
        (
            "turn_prob_dist_ylabel",
            "--turn-prob-dist-ylabel",
        ),
        (
            "corr_sli_vs_rpt_xlabel",
            "--corr-sli-vs-rpt-xlabel",
        ),
        (
            "corr_sli_vs_rpt_ylabel",
            "--corr-sli-vs-rpt-ylabel",
        ),
        (
            "corr_pre_reward_pi_vs_sli_xlabel",
            "--corr-pre-reward-pi-vs-sli-xlabel",
        ),
        (
            "corr_pre_reward_pi_vs_sli_ylabel",
            "--corr-pre-reward-pi-vs-sli-ylabel",
        ),
        (
            "corr_pre_floor_exploration_vs_sli_xlabel",
            "--corr-pre-floor-exploration-vs-sli-xlabel",
        ),
        (
            "corr_pre_floor_exploration_vs_sli_ylabel",
            "--corr-pre-floor-exploration-vs-sli-ylabel",
        ),
        (
            "corr_fast_vs_strong_xlabel",
            "--corr-fast-vs-strong-xlabel",
        ),
        (
            "corr_fast_vs_strong_ylabel",
            "--corr-fast-vs-strong-ylabel",
        ),
    ):
        value = styled_item.get(item_key)
        if value is not None:
            set_command_flag(tokens, flag, value)
    styled_item["command"] = shlex.join(tokens)

    if "output_path" in styled_item:
        styled_item["output_path"] = replace_image_suffix(
            styled_item["output_path"], style["image_format"]
        )

    if "output_paths" in styled_item:
        styled_item["output_paths"] = [
            replace_image_suffix(path, style["image_format"])
            for path in styled_item["output_paths"]
        ]

    if "rename_outputs" in styled_item:
        styled_item["rename_outputs"] = [
            {
                **rename,
                "from": replace_image_suffix(rename["from"], style["image_format"]),
                "to": replace_image_suffix(rename["to"], style["image_format"]),
            }
            for rename in styled_item["rename_outputs"]
        ]

    return styled_item


def resolve_stage_commands(commands):
    return [apply_plot_style_to_item(item) for item in commands]


def show_command_block(title, command):
    display(Markdown(f"**{title}**"))
    display(Markdown(f"```bash\n{command}\n```"))


def show_output_artifact(item):
    output_paths = item.get("output_paths")
    if output_paths is None:
        output_path = item.get("output_path")
        output_paths = [output_path] if output_path else []

    for output_path in output_paths:
        artifact = ROOT / output_path
        if not artifact.exists():
            display(Markdown(f"_Expected output not found: `{artifact}`_"))
            continue

        display(Markdown(f"Saved output: `{artifact}`"))
        if artifact.suffix.lower() in {".png", ".jpg", ".jpeg", ".gif"}:
            display(Image(filename=str(artifact)))


def apply_output_renames(item):
    for rename in item.get("rename_outputs", []):
        src = ROOT / rename["from"]
        dst = ROOT / rename["to"]
        if not src.exists():
            raise FileNotFoundError(
                f"Expected output to rename was not found: {src}"
            )
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(src), str(dst))
        display(Markdown(f"Renamed output: `{src}` -> `{dst}`"))


def preview_stage(stage_name, commands):
    commands = resolve_stage_commands(commands)
    display(Markdown(f"### {stage_name}"))
    for item in commands:
        show_command_block(item["label"], item["command"])
        for rename in item.get("rename_outputs", []):
            src = ROOT / rename["from"]
            dst = ROOT / rename["to"]
            display(Markdown(f"Rename after run: `{src}` -> `{dst}`"))
        show_output_artifact(item)


def run_stage(stage_name, commands, run=False):
    commands = resolve_stage_commands(commands)
    display(Markdown(f"### {stage_name}"))
    if not run:
        display(Markdown("_Preview mode only. Set the stage toggle to `True` to execute._"))
        for item in commands:
            show_command_block(item["label"], item["command"])
            for rename in item.get("rename_outputs", []):
                src = ROOT / rename["from"]
                dst = ROOT / rename["to"]
                display(Markdown(f"Rename after run: `{src}` -> `{dst}`"))
            show_output_artifact(item)
        return

    for item in commands:
        show_command_block(item["label"], item["command"])
        completed = subprocess.run(item["command"], shell=True, cwd=ROOT)
        if completed.returncode != 0:
            raise RuntimeError(
                f"Command failed for '{item['label']}' with exit code {completed.returncode}."
            )
        apply_output_renames(item)
        show_output_artifact(item)


## Panel 1

### Panel 1 Plot Description

Panel for **binned return probability** using excursion bins **2-8 mm**, **8-16 mm**, and **16-24 mm**.

This prototype compares:

- Intact Control>Kir
- Intact PFN>Kir
- AR Control>Kir

In [ ]:
panel_1 = {
    "title": "Panel 1 - Binned return probability",
    "description": (
        "Placeholder panel for binned return probability using 2-8 mm, 8-16 mm, "
        "and 16-24 mm excursion bins."
    ),
    "export_commands": [
        {
            "label": "Intact Control>Kir",
            "command": "python analyze.py -v '/media/Synology4/Yang Chen/2025-05-30/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-05-30/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-06-02/c5[12]_*,/media/Synology4/Yang Chen/2025-06-29/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-06-30/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-06-30/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-0[367]/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-07-0[367]/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-0[45]/c5[12]_*,/media/Synology4/Yang Chen/2025-07-05/c6[12]_*,/media/Synology4/Yang Chen/2025-07-11/c5[12]_*,/media/Synology4/Yang Chen/2025-07-11/c6[12]_*,/media/Synology4/Yang Chen/2025-07-26/afternoon/c3[12]_*,/media/Synology4/Yang Chen/2025-07-26/afternoon/c6[12]_*' -f 0-1 --rCC 15 --export-return-prob-excursion-bin-sli-bundle exports/ret_prob_binned_intact_ctrlKir_flatLgc_T2_2026-04-07.npz --return-prob-excursion-bin-trainings 2 --return-prob-excursion-bin-edges-mm 2,8,16,24 --best-worst-trn 2 --sli-use-training-mean --sli-select-skip-first-sync-buckets 1 --export-group-label \"Intact Control>Kir\""
        },
        {
            "label": "Intact PFN>Kir",
            "command": "python analyze.py -v '/media/Synology4/Yang Chen/2025-05-31/c5[12]_*,/media/Synology4/Yang Chen/2025-05-31/c6[12]_*,/media/Synology4/Yang Chen/2025-05-31/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-06-05/c6[12]_*,/media/Synology4/Yang Chen/2025-06-28/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-06-29/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-0[124]/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-07-0[124]/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-06/c6[12]_*,/media/Synology4/Yang Chen/2025-07-12/night/c32_*,/media/Synology4/Yang Chen/2025-07-21/night/c5[12]_*,/media/Synology4/Yang Chen/2025-07-22/night/c3[12]_*' -f 0-1 --rCC 15 --export-return-prob-excursion-bin-sli-bundle exports/ret_prob_binned_intact_pfnKir_flatLgc_T2_2026-04-07.npz --return-prob-excursion-bin-trainings 2 --return-prob-excursion-bin-edges-mm 2,8,16,24 --best-worst-trn 2 --sli-use-training-mean --sli-select-skip-first-sync-buckets 1 --export-group-label \"Intact PFN>Kir\""
        },
        {
            "label": "AR Control>Kir",
            "command": "python analyze.py -v '/media/Synology4/Yang Chen/2025-07-17/c4[12]_*,/media/Synology4/Yang Chen/2025-07-17/c5[12]_*,/media/Synology4/Yang Chen/2025-07-17/c6[12]_*,/media/Synology4/Yang Chen/2025-07-18/night/c5[12]_*,/media/Synology4/Yang Chen/2025-07-18/night/c6[12]_*,/media/Synology4/Yang Chen/2025-07-2[09]/night/c3[12]_*,/media/Synology4/Yang Chen/2025-07-2[01]/night/c4[12]_*,/media/Synology4/Yang Chen/2025-07-21/night/c6[12]_*,/media/Synology4/Yang Chen/2025-07-27/c6[12]_*,/media/Synology4/Yang Chen/2025-07-27/c5[12]_*,/media/Synology4/Yang Chen/2025-07-27/afternoon/c3[12]_*,/media/Synology4/Yang Chen/2025-07-27/afternoon/c4[12]_*,/media/Synology4/Yang Chen/2025-07-2[79]/night/c4[12]_*,/media/Synology4/Yang Chen/2025-07-30/night/c3[12]_*,/media/Synology4/Yang Chen/2025-07-30/night/c4[12]_*,/media/Synology4/Yang Chen/2025-08-02/night/c3[12]_*' -f 0-1 --rCC 15 --export-return-prob-excursion-bin-sli-bundle exports/ret_prob_binned_ar_ctrlKir_flatLgc_T2_2026-04-07.npz --return-prob-excursion-bin-trainings 2 --return-prob-excursion-bin-edges-mm 2,8,16,24 --best-worst-trn 2 --sli-use-training-mean --sli-select-skip-first-sync-buckets 1 --export-group-label \"AR Control>Kir\""
        },
        {
            "label": "AR PFN>Kir",
            "command": "python analyze.py -v '/media/Synology4/Yang Chen/2025-07-1[24]/afternoon/c3[12]_*,/media/Synology4/Yang Chen/2025-07-12/afternoon/c4[12]_*,/media/Synology4/Yang Chen/2025-07-14/c4[12]_*,/media/Synology4/Yang Chen/2025-07-14/c5[12]_*,/media/Synology4/Yang Chen/2025-07-14/c6[12]_*,/media/Synology4/Yang Chen/2025-07-14/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-07-14/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-15/night/c3[12]_*,/media/Synology4/Yang Chen/2025-07-15/night/c4[12]_*,/media/Synology4/Yang Chen/2025-07-15/night/c5[12]_*,/media/Synology4/Yang Chen/2025-07-15/night/c6[12]_*' -f 0-1 --rCC 15 --export-return-prob-excursion-bin-sli-bundle exports/ret_prob_binned_ar_pfnKir_flatLgc_T2_2026-04-07.npz --return-prob-excursion-bin-trainings 2 --return-prob-excursion-bin-edges-mm 2,8,16,24 --best-worst-trn 2 --sli-use-training-mean --sli-select-skip-first-sync-buckets 1 --export-group-label \"AR PFN>Kir\""
        },
    ],
    "plot_commands": [
        {
            "label": "comparison: Intact Control>Kir vs Intact PFN>Kir vs AR Control>Kir",
            "command": "python -m scripts.plot_return_prob_excursion_bin_sli_bundles --bundles \"exports/ret_prob_binned_intact_ctrlKir_flatLgc_T2_2026-04-07.npz,exports/ret_prob_binned_intact_pfnKir_flatLgc_T2_2026-04-07.npz,exports/ret_prob_binned_ar_ctrlKir_flatLgc_T2_2026-04-07.npz\" --out exports/ret_prob_binned_intact_ctrlKir_vs_intact_pfnKir_vs_ar_ctrlKir_flatLgc_T2_2026-04-07.png --stats",
            "xlabel": None,
            "ylabel": None,
            "output_path": "exports/ret_prob_binned_intact_ctrlKir_vs_intact_pfnKir_vs_ar_ctrlKir_flatLgc_T2_2026-04-07.png"
        }
    ],
}

panel_1

### Panel 1 Data Export Commands

In [ ]:
RUN_PANEL_1_EXPORTS = False
run_stage("Panel 1: export bundles", panel_1["export_commands"], run=RUN_PANEL_1_EXPORTS)

### Panel 1 Plotting Commands

In [ ]:
RUN_PANEL_1_PLOTS = False
run_stage("Panel 1: generate plot", panel_1["plot_commands"], run=RUN_PANEL_1_PLOTS)

## Panel 2

### Panel 2 Plot Description

Panel for **binned return probability** using excursion bin **20-24 mm**.

This prototype compares:

- Intact Control>Kir
- Intact PFN>Kir
- AR Control>Kir

In [ ]:
panel_2 = {
    "title": "Panel 2 - Binned return probability (20-24 mm detail)",
    "description": (
        "Detail variant of panel 1 for binned return probability using only the "
        "20-24 mm excursion bin."
    ),
    "export_commands": [
        {
            "label": "Intact Control>Kir",
            "command": "python analyze.py -v '/media/Synology4/Yang Chen/2025-05-30/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-05-30/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-06-02/c5[12]_*,/media/Synology4/Yang Chen/2025-06-29/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-06-30/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-06-30/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-0[367]/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-07-0[367]/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-0[45]/c5[12]_*,/media/Synology4/Yang Chen/2025-07-05/c6[12]_*,/media/Synology4/Yang Chen/2025-07-11/c5[12]_*,/media/Synology4/Yang Chen/2025-07-11/c6[12]_*,/media/Synology4/Yang Chen/2025-07-26/afternoon/c3[12]_*,/media/Synology4/Yang Chen/2025-07-26/afternoon/c6[12]_*' -f 0-1 --rCC 15 --export-return-prob-excursion-bin-sli-bundle exports/ret_prob_binned_intact_ctrlKir_flatLgc_T2_20mmMin24mmMax_2026-04-08.npz --return-prob-excursion-bin-trainings 2 --return-prob-excursion-bin-edges-mm 20,24 --best-worst-trn 2 --sli-use-training-mean --sli-select-skip-first-sync-buckets 1 --export-group-label \"Intact Control>Kir\""
        },
        {
            "label": "Intact PFN>Kir",
            "command": "python analyze.py -v '/media/Synology4/Yang Chen/2025-05-31/c5[12]_*,/media/Synology4/Yang Chen/2025-05-31/c6[12]_*,/media/Synology4/Yang Chen/2025-05-31/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-06-05/c6[12]_*,/media/Synology4/Yang Chen/2025-06-28/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-06-29/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-0[124]/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-07-0[124]/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-06/c6[12]_*,/media/Synology4/Yang Chen/2025-07-12/night/c32_*,/media/Synology4/Yang Chen/2025-07-21/night/c5[12]_*,/media/Synology4/Yang Chen/2025-07-22/night/c3[12]_*' -f 0-1 --rCC 15 --export-return-prob-excursion-bin-sli-bundle exports/ret_prob_binned_intact_pfnKir_flatLgc_T2_20mmMin24mmMax_2026-04-08.npz --return-prob-excursion-bin-trainings 2 --return-prob-excursion-bin-edges-mm 20,24 --best-worst-trn 2 --sli-use-training-mean --sli-select-skip-first-sync-buckets 1 --export-group-label \"Intact PFN>Kir\""
        },
        {
            "label": "AR Control>Kir",
            "command": "python analyze.py -v '/media/Synology4/Yang Chen/2025-07-17/c4[12]_*,/media/Synology4/Yang Chen/2025-07-17/c5[12]_*,/media/Synology4/Yang Chen/2025-07-17/c6[12]_*,/media/Synology4/Yang Chen/2025-07-18/night/c5[12]_*,/media/Synology4/Yang Chen/2025-07-18/night/c6[12]_*,/media/Synology4/Yang Chen/2025-07-2[09]/night/c3[12]_*,/media/Synology4/Yang Chen/2025-07-2[01]/night/c4[12]_*,/media/Synology4/Yang Chen/2025-07-21/night/c6[12]_*,/media/Synology4/Yang Chen/2025-07-27/c6[12]_*,/media/Synology4/Yang Chen/2025-07-27/c5[12]_*,/media/Synology4/Yang Chen/2025-07-27/afternoon/c3[12]_*,/media/Synology4/Yang Chen/2025-07-27/afternoon/c4[12]_*,/media/Synology4/Yang Chen/2025-07-2[79]/night/c4[12]_*,/media/Synology4/Yang Chen/2025-07-30/night/c3[12]_*,/media/Synology4/Yang Chen/2025-07-30/night/c4[12]_*,/media/Synology4/Yang Chen/2025-08-02/night/c3[12]_*' -f 0-1 --rCC 15 --export-return-prob-excursion-bin-sli-bundle exports/ret_prob_binned_ar_ctrlKir_flatLgc_T2_20mmMin24mmMax_2026-04-08.npz --return-prob-excursion-bin-trainings 2 --return-prob-excursion-bin-edges-mm 20,24 --best-worst-trn 2 --sli-use-training-mean --sli-select-skip-first-sync-buckets 1 --export-group-label \"AR Control>Kir\""
        },
        {
            "label": "AR PFN>Kir",
            "command": "python analyze.py -v '/media/Synology4/Yang Chen/2025-07-1[24]/afternoon/c3[12]_*,/media/Synology4/Yang Chen/2025-07-12/afternoon/c4[12]_*,/media/Synology4/Yang Chen/2025-07-14/c4[12]_*,/media/Synology4/Yang Chen/2025-07-14/c5[12]_*,/media/Synology4/Yang Chen/2025-07-14/c6[12]_*,/media/Synology4/Yang Chen/2025-07-14/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-07-14/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-15/night/c3[12]_*,/media/Synology4/Yang Chen/2025-07-15/night/c4[12]_*,/media/Synology4/Yang Chen/2025-07-15/night/c5[12]_*,/media/Synology4/Yang Chen/2025-07-15/night/c6[12]_*' -f 0-1 --rCC 15 --export-return-prob-excursion-bin-sli-bundle exports/ret_prob_binned_ar_pfnKir_flatLgc_T2_20mmMin24mmMax_2026-04-08.npz --return-prob-excursion-bin-trainings 2 --return-prob-excursion-bin-edges-mm 20,24 --best-worst-trn 2 --sli-use-training-mean --sli-select-skip-first-sync-buckets 1 --export-group-label \"AR PFN>Kir\""
        },
    ],
    "plot_commands": [
        {
            "label": "comparison: Intact Control>Kir vs Intact PFN>Kir vs AR Control>Kir",
            "command": "python -m scripts.plot_return_prob_excursion_bin_sli_bundles --bundles \"exports/ret_prob_binned_intact_ctrlKir_flatLgc_T2_20mmMin24mmMax_2026-04-08.npz,exports/ret_prob_binned_intact_pfnKir_flatLgc_T2_20mmMin24mmMax_2026-04-08.npz,exports/ret_prob_binned_ar_ctrlKir_flatLgc_T2_20mmMin24mmMax_2026-04-08.npz\" --out exports/ret_prob_binned_intact_ctrlKir_vs_intact_pfnKir_vs_ar_ctrlKir_flatLgc_T2_20mmMin24mmMax_2026-04-08.png --stats",
            "xlabel": None,
            "ylabel": None,
            "output_path": "exports/ret_prob_binned_intact_ctrlKir_vs_intact_pfnKir_vs_ar_ctrlKir_flatLgc_T2_20mmMin24mmMax_2026-04-08.png"
        }
    ],
}

panel_2

### Panel 2 Data Export Commands

In [ ]:
RUN_PANEL_2_EXPORTS = False
run_stage("Panel 2: export bundles", panel_2["export_commands"], run=RUN_PANEL_2_EXPORTS)

### Panel 2 Plotting Commands

In [ ]:
RUN_PANEL_2_PLOTS = False
run_stage("Panel 2: generate plot", panel_2["plot_commands"], run=RUN_PANEL_2_PLOTS)

## Panel 3

### Panel 3 Plot Description

Panel for **Between-Reward Distance Traveled, T2** using **short (0-60)**, **mid-range (60-200)**, and **long (200-1600)** distance bins between rewards.

This panel compares:

- Intact Control>Kir
- Intact PFN>Kir
- AR Control>Kir

The intended interpretation is that AR Control>Kir flies show a high prevalence of short-distance trajectories, Intact PFN>Kir flies dominate the mid-range distances, and both perturbed groups outweigh Intact Control>Kir flies for long-distance travel between rewards.

In [ ]:
panel_3 = {
    "title": "Panel 3 - Between-Reward Distance Traveled, T2",
    "description": (
        "Compares Intact Control>Kir, Intact PFN>Kir, and AR "
        "Control>Kir flies across short, mid-range, and long between-reward "
        "distance bins during training 2."
    ),
    "export_commands": [
        {
            "label": "Intact Control>Kir",
            "command": "python analyze.py -v \"/media/Synology4/Yang Chen/2025-05-30/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-05-30/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-06-02/c5[12]_*,/media/Synology4/Yang Chen/2025-06-29/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-06-30/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-06-30/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-0[367]/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-07-0[367]/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-0[45]/c5[12]_*,/media/Synology4/Yang Chen/2025-07-05/c6[12]_*,/media/Synology4/Yang Chen/2025-07-11/c5[12]_*,/media/Synology4/Yang Chen/2025-07-11/c6[12]_*,/media/Synology4/Yang Chen/2025-07-26/afternoon/c3[12]_*,/media/Synology4/Yang Chen/2025-07-26/afternoon/c6[12]_*\" -f 0-1 --rCC 15 --btw-rwd-dist-hist --btw-rwd-dist-per-fly --btw-rwd-dist-bin-edges \"0,60,200,1600\" --btw-rwd-dist-normalize --btw-rwd-dist-trainings 2 --btw-rwd-dist-ci --skip-first-sync-buckets 1 --btw-rwd-dist-export-hist exports/dist_trav_intact_ctrlKir_flatLgc_T2_inclWall_2026-04-08.npz"
        },
        {
            "label": "Intact PFN>Kir",
            "command": "python analyze.py -v '/media/Synology4/Yang Chen/2025-05-31/c5[12]_*,/media/Synology4/Yang Chen/2025-05-31/c6[12]_*,/media/Synology4/Yang Chen/2025-05-31/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-06-05/c6[12]_*,/media/Synology4/Yang Chen/2025-06-28/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-06-29/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-0[124]/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-07-0[124]/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-06/c6[12]_*,/media/Synology4/Yang Chen/2025-07-12/night/c32_*,/media/Synology4/Yang Chen/2025-07-21/night/c5[12]_*,/media/Synology4/Yang Chen/2025-07-22/night/c3[12]_*' -f 0-1 --rCC 15 --btw-rwd-dist-hist --btw-rwd-dist-per-fly --btw-rwd-dist-bin-edges \"0,60,200,1600\" --btw-rwd-dist-normalize --btw-rwd-dist-trainings 2 --btw-rwd-dist-ci --skip-first-sync-buckets 1 --btw-rwd-dist-export-hist exports/dist_trav_intact_pfnKir_flatLgc_T2_inclWall_2026-04-08.npz"
        },
        {
            "label": "AR Control>Kir",
            "command": "python analyze.py -v '/media/Synology4/Yang Chen/2025-07-17/c4[12]_*,/media/Synology4/Yang Chen/2025-07-17/c5[12]_*,/media/Synology4/Yang Chen/2025-07-17/c6[12]_*,/media/Synology4/Yang Chen/2025-07-18/night/c5[12]_*,/media/Synology4/Yang Chen/2025-07-18/night/c6[12]_*,/media/Synology4/Yang Chen/2025-07-2[09]/night/c3[12]_*,/media/Synology4/Yang Chen/2025-07-2[01]/night/c4[12]_*,/media/Synology4/Yang Chen/2025-07-21/night/c6[12]_*,/media/Synology4/Yang Chen/2025-07-27/c6[12]_*,/media/Synology4/Yang Chen/2025-07-27/c5[12]_*,/media/Synology4/Yang Chen/2025-07-27/afternoon/c3[12]_*,/media/Synology4/Yang Chen/2025-07-27/afternoon/c4[12]_*,/media/Synology4/Yang Chen/2025-07-2[79]/night/c4[12]_*,/media/Synology4/Yang Chen/2025-07-30/night/c3[12]_*,/media/Synology4/Yang Chen/2025-07-30/night/c4[12]_*,/media/Synology4/Yang Chen/2025-08-02/night/c3[12]_*' -f 0-1 --rCC 15 --btw-rwd-dist-hist --btw-rwd-dist-per-fly --btw-rwd-dist-bin-edges \"0,60,200,1600\" --btw-rwd-dist-normalize --btw-rwd-dist-trainings 2 --btw-rwd-dist-ci --skip-first-sync-buckets 1 --btw-rwd-dist-export-hist exports/dist_trav_ar_ctrlKir_flatLgc_T2_inclWall_2026-04-08.npz"
        },
    ],
    "plot_commands": [
        {
            "label": "comparison: Intact Control>Kir vs Intact PFN>Kir vs AR Control>Kir",
            "command": "python -m scripts.plot_overlay_training_metric_hist --input \"Intact Control>Kir=exports/dist_trav_intact_ctrlKir_flatLgc_T2_inclWall_2026-04-08.npz\" --input \"Intact PFN>Kir=exports/dist_trav_intact_pfnKir_flatLgc_T2_inclWall_2026-04-08.npz\" --input \"AR Control>Kir=exports/dist_trav_ar_ctrlKir_flatLgc_T2_inclWall_2026-04-08.npz\" --out exports/dist_trav_intact_ctrlKir_vs_intact_pfnKir_vs_ar_ctrlKir_flatLgc_T2_inclWall_2026-04-08.png --stats",
            "xlabel": None,
            "ylabel": None,
            "output_path": "exports/dist_trav_intact_ctrlKir_vs_intact_pfnKir_vs_ar_ctrlKir_flatLgc_T2_inclWall_2026-04-08.png"
        }
    ],
}

panel_3

### Panel 3 Data Export Commands

In [ ]:
RUN_PANEL_3_EXPORTS = False
run_stage("Panel 3: export bundles", panel_3["export_commands"], run=RUN_PANEL_3_EXPORTS)

### Panel 3 Plotting Commands

In [ ]:
RUN_PANEL_3_PLOTS = False
run_stage("Panel 3: generate plot", panel_3["plot_commands"], run=RUN_PANEL_3_PLOTS)

## Panel 4

### Panel 4 Plot Description

Panel for **Between-Reward Distance Traveled vs Max Distance from Reward (T1, wall contact included)** using **2-8 mm**, **8-16 mm**, and **16-24 mm** bins for maximum distance from reward.

This panel compares:

- Intact Control>Kir
- Intact PFN>Kir
- AR Control>Kir

The intended interpretation is that, for the smallest max-distance bin, intact controls have a significantly larger mean distance traveled, potentially reflecting more frequent loop movements rather than accidental re-entries. For the mid-range bin, Intact PFN>Kir flies show significantly higher distance traveled, consistent with circling the reward without reaching far-range distances. For the far-range bin, AR Control>Kir flies show significantly higher distance traveled, apparently driven largely by between-reward segments that involve wall contact.

In [ ]:
panel_4 = {
    "title": "Panel 4 - Between-Reward Distance Traveled vs Max Distance from Reward (T1, wall contact included)",
    "description": (
        "Compares Intact Control>Kir, Intact PFN>Kir, and AR "
        "Control>Kir flies for between-reward distance traveled conditioned on "
        "the segment's maximum distance from reward during training 1, with wall-contact "
        "segments included."
    ),
    "export_commands": [
        {
            "label": "Intact Control>Kir",
            "command": "python analyze.py -v \"/media/Synology4/Yang Chen/2025-05-30/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-05-30/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-06-02/c5[12]_*,/media/Synology4/Yang Chen/2025-06-29/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-06-30/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-06-30/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-0[367]/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-07-0[367]/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-0[45]/c5[12]_*,/media/Synology4/Yang Chen/2025-07-05/c6[12]_*,/media/Synology4/Yang Chen/2025-07-11/c5[12]_*,/media/Synology4/Yang Chen/2025-07-11/c6[12]_*,/media/Synology4/Yang Chen/2025-07-26/afternoon/c3[12]_*,/media/Synology4/Yang Chen/2025-07-26/afternoon/c6[12]_*\" -f 0-1 --rCC 15 --btw-rwd-conditioned-disttrav --btw-rwd-conditioned-disttrav-trn 1 --btw-rwd-conditioned-disttrav-skip-first-sync-buckets 1 --btw-rwd-conditioned-disttrav-bin-edges 2,8,16,24 --btw-rwd-conditioned-disttrav-export-npz exports/disttrav_by_dist_max_bin_intact_ctrlKir_flatLgc_T1_inclWall_2026-04-08.npz"
        },
        {
            "label": "Intact PFN>Kir",
            "command": "python analyze.py -v '/media/Synology4/Yang Chen/2025-05-31/c5[12]_*,/media/Synology4/Yang Chen/2025-05-31/c6[12]_*,/media/Synology4/Yang Chen/2025-05-31/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-06-05/c6[12]_*,/media/Synology4/Yang Chen/2025-06-28/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-06-29/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-0[124]/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-07-0[124]/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-06/c6[12]_*,/media/Synology4/Yang Chen/2025-07-12/night/c32_*,/media/Synology4/Yang Chen/2025-07-21/night/c5[12]_*,/media/Synology4/Yang Chen/2025-07-22/night/c3[12]_*' -f 0-1 --rCC 15 --btw-rwd-conditioned-disttrav --btw-rwd-conditioned-disttrav-trn 1 --btw-rwd-conditioned-disttrav-skip-first-sync-buckets 1 --btw-rwd-conditioned-disttrav-bin-edges 2,8,16,24 --btw-rwd-conditioned-disttrav-export-npz exports/disttrav_by_dist_max_bin_intact_pfnKir_flatLgc_T1_inclWall_2026-04-08.npz"
        },
        {
            "label": "AR Control>Kir",
            "command": "python analyze.py -v '/media/Synology4/Yang Chen/2025-07-17/c4[12]_*,/media/Synology4/Yang Chen/2025-07-17/c5[12]_*,/media/Synology4/Yang Chen/2025-07-17/c6[12]_*,/media/Synology4/Yang Chen/2025-07-18/night/c5[12]_*,/media/Synology4/Yang Chen/2025-07-18/night/c6[12]_*,/media/Synology4/Yang Chen/2025-07-2[09]/night/c3[12]_*,/media/Synology4/Yang Chen/2025-07-2[01]/night/c4[12]_*,/media/Synology4/Yang Chen/2025-07-21/night/c6[12]_*,/media/Synology4/Yang Chen/2025-07-27/c6[12]_*,/media/Synology4/Yang Chen/2025-07-27/c5[12]_*,/media/Synology4/Yang Chen/2025-07-27/afternoon/c3[12]_*,/media/Synology4/Yang Chen/2025-07-27/afternoon/c4[12]_*,/media/Synology4/Yang Chen/2025-07-2[79]/night/c4[12]_*,/media/Synology4/Yang Chen/2025-07-30/night/c3[12]_*,/media/Synology4/Yang Chen/2025-07-30/night/c4[12]_*,/media/Synology4/Yang Chen/2025-08-02/night/c3[12]_*' -f 0-1 --rCC 15 --btw-rwd-conditioned-disttrav --btw-rwd-conditioned-disttrav-trn 1 --btw-rwd-conditioned-disttrav-skip-first-sync-buckets 1 --btw-rwd-conditioned-disttrav-bin-edges 2,8,16,24 --btw-rwd-conditioned-disttrav-export-npz exports/disttrav_by_dist_max_bin_ar_ctrlKir_flatLgc_T1_inclWall_2026-04-08.npz"
        },
    ],
    "plot_commands": [
        {
            "label": "comparison: Intact Control>Kir vs Intact PFN>Kir vs AR Control>Kir",
            "command": "python analyze.py -v \"/media/Synology4/Yang Chen/2025-05-31/c5[1]_*\" -f 0-1 --rCC 15 --btw-rwd-conditioned-disttrav --btw-rwd-conditioned-disttrav-import-npz \"Intact Control>Kir:exports/disttrav_by_dist_max_bin_intact_ctrlKir_flatLgc_T1_inclWall_2026-04-08.npz\" --btw-rwd-conditioned-disttrav-import-npz \"Intact PFN>Kir:exports/disttrav_by_dist_max_bin_intact_pfnKir_flatLgc_T1_inclWall_2026-04-08.npz\" --btw-rwd-conditioned-disttrav-import-npz \"AR Control>Kir:exports/disttrav_by_dist_max_bin_ar_ctrlKir_flatLgc_T1_inclWall_2026-04-08.npz\" --btw-rwd-conditioned-disttrav-stats --btw-rwd-conditioned-disttrav-import-out exports/disttrav_by_dist_max_bin_intact_ctrlKir_vs_intact_pfnKir_vs_ar_ctrlKir_T1_inclWall_2026-04-08.png",
            "btw_rwd_conditioned_disttrav_xlabel": None,
            "btw_rwd_conditioned_disttrav_ylabel": None,
            "output_paths": [
                "exports/disttrav_by_dist_max_bin_intact_ctrlKir_vs_intact_pfnKir_vs_ar_ctrlKir_T1_inclWall_2026-04-08_total.png",
                "exports/disttrav_by_dist_max_bin_intact_ctrlKir_vs_intact_pfnKir_vs_ar_ctrlKir_T1_inclWall_2026-04-08_tail.png"
            ]
        }
    ],
}

panel_4

### Panel 4 Data Export Commands

In [ ]:
RUN_PANEL_4_EXPORTS = False
run_stage("Panel 4: export bundles", panel_4["export_commands"], run=RUN_PANEL_4_EXPORTS)

### Panel 4 Plotting Commands

In [ ]:
RUN_PANEL_4_PLOTS = False
run_stage("Panel 4: generate plot", panel_4["plot_commands"], run=RUN_PANEL_4_PLOTS)

## Panel 5

### Panel 5 Plot Description

Panel for **Between-Reward Distance Traveled vs Max Distance from Reward (T1, wall contact included, 20-24 mm detail)** focused on the subset of between-reward segments whose maximum distance from the reward circle falls between **20 mm** and **24 mm**.

This panel compares:

- Intact Control>Kir
- Intact PFN>Kir
- AR Control>Kir

The intended interpretation is that this is a detail view of panel 4's far-range regime, where AR Control>Kir flies show the highest between-reward distance traveled.

In [ ]:
panel_5 = {
    "title": "Panel 5 - Between-Reward Distance Traveled vs Max Distance from Reward (T1, wall contact included, 20-24 mm detail)",
    "description": (
        "Shows the 20-24 mm subset of panel 4, comparing Intact Control>Kir, "
        "Intact PFN>Kir, and AR Control>Kir flies for between-reward "
        "distance traveled conditioned on the segment's maximum distance from reward "
        "during training 1, with wall-contact segments included."
    ),
    "export_commands": [
        {
            "label": "Intact Control>Kir",
            "command": "python analyze.py -v \"/media/Synology4/Yang Chen/2025-05-30/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-05-30/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-06-02/c5[12]_*,/media/Synology4/Yang Chen/2025-06-29/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-06-30/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-06-30/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-0[367]/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-07-0[367]/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-0[45]/c5[12]_*,/media/Synology4/Yang Chen/2025-07-05/c6[12]_*,/media/Synology4/Yang Chen/2025-07-11/c5[12]_*,/media/Synology4/Yang Chen/2025-07-11/c6[12]_*,/media/Synology4/Yang Chen/2025-07-26/afternoon/c3[12]_*,/media/Synology4/Yang Chen/2025-07-26/afternoon/c6[12]_*\" -f 0-1 --rCC 15 --btw-rwd-conditioned-disttrav --btw-rwd-conditioned-disttrav-trn 1 --btw-rwd-conditioned-disttrav-skip-first-sync-buckets 1 --btw-rwd-conditioned-disttrav-bin-edges 20,24 --btw-rwd-conditioned-disttrav-export-npz exports/disttrav_by_dist_max_bin_intact_ctrlKir_flatLgc_T1_inclWall_20mmMin24mmMax_2026-04-08.npz"
        },
        {
            "label": "Intact PFN>Kir",
            "command": "python analyze.py -v '/media/Synology4/Yang Chen/2025-05-31/c5[12]_*,/media/Synology4/Yang Chen/2025-05-31/c6[12]_*,/media/Synology4/Yang Chen/2025-05-31/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-06-05/c6[12]_*,/media/Synology4/Yang Chen/2025-06-28/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-06-29/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-0[124]/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-07-0[124]/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-06/c6[12]_*,/media/Synology4/Yang Chen/2025-07-12/night/c32_*,/media/Synology4/Yang Chen/2025-07-21/night/c5[12]_*,/media/Synology4/Yang Chen/2025-07-22/night/c3[12]_*' -f 0-1 --rCC 15 --btw-rwd-conditioned-disttrav --btw-rwd-conditioned-disttrav-trn 1 --btw-rwd-conditioned-disttrav-skip-first-sync-buckets 1 --btw-rwd-conditioned-disttrav-bin-edges 20,24 --btw-rwd-conditioned-disttrav-export-npz exports/disttrav_by_dist_max_bin_intact_pfnKir_flatLgc_T1_inclWall_20mmMin24mmMax_2026-04-08.npz"
        },
        {
            "label": "AR Control>Kir",
            "command": "python analyze.py -v '/media/Synology4/Yang Chen/2025-07-17/c4[12]_*,/media/Synology4/Yang Chen/2025-07-17/c5[12]_*,/media/Synology4/Yang Chen/2025-07-17/c6[12]_*,/media/Synology4/Yang Chen/2025-07-18/night/c5[12]_*,/media/Synology4/Yang Chen/2025-07-18/night/c6[12]_*,/media/Synology4/Yang Chen/2025-07-2[09]/night/c3[12]_*,/media/Synology4/Yang Chen/2025-07-2[01]/night/c4[12]_*,/media/Synology4/Yang Chen/2025-07-21/night/c6[12]_*,/media/Synology4/Yang Chen/2025-07-27/c6[12]_*,/media/Synology4/Yang Chen/2025-07-27/c5[12]_*,/media/Synology4/Yang Chen/2025-07-27/afternoon/c3[12]_*,/media/Synology4/Yang Chen/2025-07-27/afternoon/c4[12]_*,/media/Synology4/Yang Chen/2025-07-2[79]/night/c4[12]_*,/media/Synology4/Yang Chen/2025-07-30/night/c3[12]_*,/media/Synology4/Yang Chen/2025-07-30/night/c4[12]_*,/media/Synology4/Yang Chen/2025-08-02/night/c3[12]_*' -f 0-1 --rCC 15 --btw-rwd-conditioned-disttrav --btw-rwd-conditioned-disttrav-trn 1 --btw-rwd-conditioned-disttrav-skip-first-sync-buckets 1 --btw-rwd-conditioned-disttrav-bin-edges 20,24 --btw-rwd-conditioned-disttrav-export-npz exports/disttrav_by_dist_max_bin_ar_ctrlKir_flatLgc_T1_inclWall_20mmMin24mmMax_2026-04-08.npz"
        },
    ],
    "plot_commands": [
        {
            "label": "comparison: Intact Control>Kir vs Intact PFN>Kir vs AR Control>Kir",
            "command": "python analyze.py -v \"/media/Synology4/Yang Chen/2025-05-31/c5[1]_*\" -f 0-1 --rCC 15 --btw-rwd-conditioned-disttrav --btw-rwd-conditioned-disttrav-import-npz \"Intact Control>Kir:exports/disttrav_by_dist_max_bin_intact_ctrlKir_flatLgc_T1_inclWall_20mmMin24mmMax_2026-04-08.npz\" --btw-rwd-conditioned-disttrav-import-npz \"Intact PFN>Kir:exports/disttrav_by_dist_max_bin_intact_pfnKir_flatLgc_T1_inclWall_20mmMin24mmMax_2026-04-08.npz\" --btw-rwd-conditioned-disttrav-import-npz \"AR Control>Kir:exports/disttrav_by_dist_max_bin_ar_ctrlKir_flatLgc_T1_inclWall_20mmMin24mmMax_2026-04-08.npz\" --btw-rwd-conditioned-disttrav-stats --btw-rwd-conditioned-disttrav-import-out exports/disttrav_by_dist_max_bin_ctrl_vs_pfnKir_vs_ar_T1_inclWall_20mmMin24mmMax_2026-04-08.png",
            "btw_rwd_conditioned_disttrav_xlabel": None,
            "btw_rwd_conditioned_disttrav_ylabel": None,
            "output_paths": [
                "exports/disttrav_by_dist_max_bin_ctrl_vs_pfnKir_vs_ar_T1_inclWall_20mmMin24mmMax_2026-04-08_total.png",
                "exports/disttrav_by_dist_max_bin_ctrl_vs_pfnKir_vs_ar_T1_inclWall_20mmMin24mmMax_2026-04-08_tail.png"
            ]
        }
    ],
}

panel_5

### Panel 5 Data Export Commands

In [ ]:
RUN_PANEL_5_EXPORTS = False
run_stage("Panel 5: export bundles", panel_5["export_commands"], run=RUN_PANEL_5_EXPORTS)

### Panel 5 Plotting Commands

In [ ]:
RUN_PANEL_5_PLOTS = False
run_stage("Panel 5: generate plot", panel_5["plot_commands"], run=RUN_PANEL_5_PLOTS)

## Panel 6

### Panel 6 Plot Description

Panel for **max distance from reward circle center** in **Intact Control>Kir** flies in the **flat large chamber**.

This panel shows the per-sync-bucket mean of maximum distance from the reward circle within flies, averaged across all between-reward segments in that time frame, and then averaged across flies to produce the mean and confidence interval.

The intended interpretation is that maximum distance is lower in **T1** and trends downward over the course of that training, consistent with the larger reward circle used there. In **T2** and later, the reward circle is smaller, so the maximum distance is larger overall and shows less improvement across the training, potentially because flies get lost more often.

This panel includes three plotting variants:

- experimental only
- experimental plus yoked controls
- top 20% vs bottom 50%

In [ ]:
panel_6 = {
    "title": "Panel 6 - Max Distance from Reward Circle Center",
    "description": (
        "Shows the per-sync-bucket mean maximum distance from the reward circle "
        "center for Intact Control>Kir flies in the flat large chamber, "
        "with within-fly averaging across between-reward segments followed by "
        "across-fly mean and confidence interval summaries."
    ),
    "export_commands": [
        {
            "label": "Intact Control>Kir",
            "command": "python analyze.py -v \"/media/Synology4/Yang Chen/2025-05-30/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-05-30/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-06-02/c5[12]_*,/media/Synology4/Yang Chen/2025-06-29/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-06-30/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-06-30/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-0[367]/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-07-0[367]/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-0[45]/c5[12]_*,/media/Synology4/Yang Chen/2025-07-05/c6[12]_*,/media/Synology4/Yang Chen/2025-07-11/c5[12]_*,/media/Synology4/Yang Chen/2025-07-11/c6[12]_*,/media/Synology4/Yang Chen/2025-07-26/afternoon/c3[12]_*,/media/Synology4/Yang Chen/2025-07-26/afternoon/c6[12]_*\" -f 0-1 --rCC 15 --export-between-reward-maxdist-sli-bundle exports/maxdist_by_sb_intact_ctrlKir_flatLgc_2026-03-24.npz --best-worst-trn 2 --sli-select-skip-first-sync-buckets 1 --sli-use-training-mean --export-group-label \"Intact Control>Kir\""
        }
    ],
    "plot_commands": [
        {
            "label": "experimental only",
            "command": "python -m scripts.plot_com_sli_bundles --bundles \"/home/tracking/code/YL/nvsl-analysis/exports/maxdist_by_sb_intact_ctrlKir_flatLgc_2026-03-24.npz\" --out exports/maxdist_by_sb_intact_ctrlKir_flatLgc_2026-03-24.png --metric between_reward_maxdist --num-trainings 2",
            "xlabel": None,
            "ylabel": None,
            "output_path": "exports/maxdist_by_sb_intact_ctrlKir_flatLgc_2026-03-24.png"
        },
        {
            "label": "experimental plus yoked controls",
            "command": "python -m scripts.plot_com_sli_bundles --bundles \"/home/tracking/code/YL/nvsl-analysis/exports/maxdist_by_sb_intact_ctrlKir_flatLgc_2026-03-24.npz\" --out exports/maxdist_by_sb_intact_ctrlKir_flatLgc_withCtrl_2026-03-24.png --metric between_reward_maxdist --include-ctrl --num-trainings 2",
            "xlabel": None,
            "ylabel": None,
            "output_path": "exports/maxdist_by_sb_intact_ctrlKir_flatLgc_withCtrl_2026-03-24.png"
        },
        {
            "label": "top 20% vs bottom 50%",
            "command": "python -m scripts.plot_com_sli_bundles --bundles \"/home/tracking/code/YL/nvsl-analysis/exports/maxdist_by_sb_intact_ctrlKir_flatLgc_2026-03-24.npz\" --out exports/maxdist_by_sb_intact_ctrlKir_flatLgc_top0.2Bot0.5_2026-03-24.png --metric between_reward_maxdist --sli-extremes both --top-sli-fraction 0.2 --bottom-sli-fraction 0.5 --num-trainings 2",
            "xlabel": None,
            "ylabel": None,
            "output_path": "exports/maxdist_by_sb_intact_ctrlKir_flatLgc_top0.2Bot0.5_2026-03-24.png"
        }
    ],
}

panel_6

### Panel 6 Data Export Commands

In [ ]:
RUN_PANEL_6_EXPORTS = False
run_stage("Panel 6: export bundles", panel_6["export_commands"], run=RUN_PANEL_6_EXPORTS)

### Panel 6 Plotting Commands

In [ ]:
RUN_PANEL_6_PLOTS = False
run_stage("Panel 6: generate plot", panel_6["plot_commands"], run=RUN_PANEL_6_PLOTS)

## Panel 7

### Panel 7 Plot Description

Panel for **return-leg distance traveled** in **Intact Control>Kir** flies in the **flat large chamber**.

This panel shows the per-sync-bucket mean of between-rewards distance traveled for the return leg of the segment, defined as the portion from when the fly reaches its maximum distance from the reward circle to when it re-enters the reward circle. Values are averaged within flies across all between-reward segments in that time frame, then averaged across flies to produce the mean and confidence interval.

The intended interpretation is that return-leg distance traveled is lower in **T1** and trends downward over the course of that training, consistent with the larger reward circle used there. In **T2** and later, the reward circle is smaller, so the return-leg distance traveled is much higher and the decrease across the training is less consistent. Notably, for the **top 20% vs bottom 50%** split, the top 20% of flies perform similarly well in both trainings.

This panel includes two plotting variants:

- experimental only
- top 20% vs bottom 50%

The experimental plus yoked-controls variant is intentionally omitted here because yoked flies travel at similar speeds and produce nearly the same result for this metric.

In [ ]:
panel_7 = {
    "title": "Panel 7 - Return-Leg Distance Traveled",
    "description": (
        "Shows the per-sync-bucket mean between-reward return-leg distance traveled "
        "for Intact Control>Kir flies in the flat large chamber, with "
        "within-fly averaging across between-reward segments followed by across-fly "
        "mean and confidence interval summaries."
    ),
    "export_commands": [
        {
            "label": "Intact Control>Kir",
            "command": "python analyze.py -v '/media/Synology4/Yang Chen/2025-05-30/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-05-30/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-06-02/c5[12]_*,/media/Synology4/Yang Chen/2025-06-29/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-06-30/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-06-30/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-0[367]/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-07-0[367]/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-0[45]/c5[12]_*,/media/Synology4/Yang Chen/2025-07-05/c6[12]_*,/media/Synology4/Yang Chen/2025-07-11/c5[12]_*,/media/Synology4/Yang Chen/2025-07-11/c6[12]_*,/media/Synology4/Yang Chen/2025-07-26/afternoon/c3[12]_*,/media/Synology4/Yang Chen/2025-07-26/afternoon/c6[12]_*' -f 0-1 --rCC 15 --export-btw-rwd-return-leg-dist-sli-bundle exports/dist_trav_return_intact_ctrlKir_flatLgc_bySB_2026-04-07.npz --export-group-label \"Intact Control>Kir\" --sli-use-training-mean --best-worst-trn 2 --sli-select-skip-first-sync-buckets 1"
        }
    ],
    "plot_commands": [
        {
            "label": "experimental only",
            "command": "python -m scripts.plot_com_sli_bundles --bundles \"/home/tracking/code/YL/nvsl-analysis/exports/dist_trav_return_intact_ctrlKir_flatLgc_bySB_2026-04-07.npz\" --out exports/dist_trav_return_intact_ctrlKir_flatLgc_bySB_2026-04-07.png --metric between_reward_return_leg_dist --num-trainings 2",
            "xlabel": None,
            "ylabel": None,
            "output_path": "exports/dist_trav_return_intact_ctrlKir_flatLgc_bySB_2026-04-07.png"
        },
        {
            "label": "top 20% vs bottom 50%",
            "command": "python -m scripts.plot_com_sli_bundles --bundles \"/home/tracking/code/YL/nvsl-analysis/exports/dist_trav_return_intact_ctrlKir_flatLgc_bySB_2026-04-07.npz\" --out exports/dist_trav_return_intact_ctrlKir_flatLgc_bySB_top0.2Bot0.5_2026-04-07.png --metric between_reward_return_leg_dist --sli-extremes both --top-sli-fraction 0.2 --bottom-sli-fraction 0.5 --num-trainings 2",
            "xlabel": None,
            "ylabel": None,
            "output_path": "exports/dist_trav_return_intact_ctrlKir_flatLgc_bySB_top0.2Bot0.5_2026-04-07.png"
        }
    ],
}

panel_7

### Panel 7 Data Export Commands

In [ ]:
RUN_PANEL_7_EXPORTS = False
run_stage("Panel 7: export bundles", panel_7["export_commands"], run=RUN_PANEL_7_EXPORTS)

### Panel 7 Plotting Commands

In [ ]:
RUN_PANEL_7_PLOTS = False
run_stage("Panel 7: generate plot", panel_7["plot_commands"], run=RUN_PANEL_7_PLOTS)

## Panel 8

### Panel 8 Plot Description

Panel for **dual-circle turnback ratio** in **Intact Control>Kir** flies in the **flat large chamber**.

This panel shows the per-sync-bucket mean of the turnback ratio. A turnback episode starts when a fly exits the inner circle around the reward location and ends when it either re-enters the inner circle, exits the outer circle, or reaches training end before either outcome is observed. Re-entry into the inner circle is counted as a successful turnback; exit from the outer circle is counted as a failed turnback; right-censored episodes are excluded from both the numerator and denominator.

The turnback ratio is computed per sync bucket as the number of episodes with re-entry into the inner circle before leaving the outer circle divided by the number of episodes with an observed outcome. Here, the inner and outer radial deltas relative to the reward circle are **4 mm** and **8 mm**, respectively.

The intended interpretation is that the experimental flies increase their turnback ratio steadily during **T1**, reaching just over **0.7**, while in the more difficult **T2** condition the ratio remains around **0.5**. Control flies stay around **0.2** in both trainings. For the **top 20% vs bottom 50%** split, both groups have similar slopes but are separated by an offset, roughly **0.2** in T1 and closer to **0.4** in T2; the top 20% group does not show much of a T2 drop.

This panel includes three plotting variants:

- experimental only
- experimental plus yoked controls
- top 20% vs bottom 50%


In [ ]:
panel_8 = {
    "title": "Panel 8 - Dual-Circle Turnback Ratio",
    "description": (
        "Shows the per-sync-bucket mean dual-circle turnback ratio for "
        "Intact Control>Kir flies in the flat large chamber, using "
        "4 mm and 8 mm inner and outer radial deltas relative to the reward circle."
    ),
    "export_commands": [
        {
            "label": "Intact Control>Kir",
            "command": 'python analyze.py -v "/media/Synology4/Yang Chen/2025-05-30/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-05-30/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-06-02/c5[12]_*,/media/Synology4/Yang Chen/2025-06-29/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-06-30/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-06-30/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-0[367]/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-07-0[367]/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-0[45]/c5[12]_*,/media/Synology4/Yang Chen/2025-07-05/c6[12]_*,/media/Synology4/Yang Chen/2025-07-11/c5[12]_*,/media/Synology4/Yang Chen/2025-07-11/c6[12]_*,/media/Synology4/Yang Chen/2025-07-26/afternoon/c3[12]_*,/media/Synology4/Yang Chen/2025-07-26/afternoon/c6[12]_*" -f 0-1 --rCC 15 --turnback-dual-circle --turnback-inner-delta-mm 4 --turnback-outer-delta-mm 8 --export-turnback-sli-bundle exports/turnback_4mm_8mm_intact_ctrlKir_flatLgc_2026-02-09.npz --export-group-label "Intact Control>Kir" --best-worst-sli --best-worst-trn 2 --sli-select-skip-first-sync-buckets 1 --sli-use-training-mean'
        }
    ],
    "plot_commands": [
        {
            "label": "experimental only",
            "command": 'python -m scripts.plot_com_sli_bundles --bundles "/home/tracking/code/YL/nvsl-analysis/exports/turnback_4mm_8mm_intact_ctrlKir_flatLgc_2026-02-09.npz" --out exports/turnback_4mm_8mm_intact_ctrlKir_flatLgc_2026-02-09.png --metric turnback --num-trainings 2',
            "xlabel": None,
            "ylabel": None,
            "output_path": "exports/turnback_4mm_8mm_intact_ctrlKir_flatLgc_2026-02-09.png"
        },
        {
            "label": "experimental plus yoked controls",
            "command": 'python -m scripts.plot_com_sli_bundles --bundles "/home/tracking/code/YL/nvsl-analysis/exports/turnback_4mm_8mm_intact_ctrlKir_flatLgc_2026-02-09.npz" --out exports/turnback_4mm_8mm_intact_ctrlKir_flatLgc_withCtrl_2026-02-09.png --metric turnback --num-trainings 2 --include-ctrl',
            "xlabel": None,
            "ylabel": None,
            "output_path": "exports/turnback_4mm_8mm_intact_ctrlKir_flatLgc_withCtrl_2026-02-09.png"
        },
        {
            "label": "top 20% vs bottom 50%",
            "command": 'python -m scripts.plot_com_sli_bundles --bundles "/home/tracking/code/YL/nvsl-analysis/exports/turnback_4mm_8mm_intact_ctrlKir_flatLgc_2026-02-09.npz" --out exports/turnback_4mm_8mm_intact_ctrlKir_flatLgc_top0.2Bot0.5_2026-02-09.png --metric turnback --num-trainings 2 --sli-extremes both --top-sli-fraction 0.2 --bottom-sli-fraction 0.5',
            "xlabel": None,
            "ylabel": None,
            "output_path": "exports/turnback_4mm_8mm_intact_ctrlKir_flatLgc_top0.2Bot0.5_2026-02-09.png"
        }
    ],
}

panel_8


### Panel 8 Data Export Commands


In [ ]:
RUN_PANEL_8_EXPORTS = False
run_stage("Panel 8: export bundles", panel_8["export_commands"], run=RUN_PANEL_8_EXPORTS)


### Panel 8 Plotting Commands


In [ ]:
RUN_PANEL_8_PLOTS = False
run_stage("Panel 8: generate plot", panel_8["plot_commands"], run=RUN_PANEL_8_PLOTS)


## Panel 9

### Panel 9 Plot Description

Panel for **COM distance to reward circle center** in **Intact Control>Kir** flies in the **flat large chamber**.

This panel shows the per-sync-bucket mean of the center-of-mass distance from the reward circle center. For each between-reward segment, the distance between the segment center of mass and the reward center is computed in millimeters. These values are then averaged across all between-reward trajectory segments in a given sync bucket for a single fly, after which the plot shows the across-fly mean and confidence interval.

This measure is used here as a rough proxy for the skewedness of the between-reward segment with respect to the reward circle center.

The intended interpretation is that the skewedness of the between-reward segments steadily decreases over **T1**, whereas in the more difficult **T2** condition, where the reward circle is smaller and harder to enter, the skewedness remains comparatively steady. Yoked control flies show skewedness values roughly an order of magnitude higher than the experimental flies in a given sync bucket, regardless of training. In the **top 20% vs bottom 50%** view, the top 20% learners maintain relatively low skewedness across both trainings.

This panel includes three plotting variants:

- experimental only
- experimental plus yoked controls
- top 20% vs bottom 50%


In [ ]:
panel_9 = {
    "title": "Panel 9 - COM Distance to Reward Circle Center",
    "description": (
        "Shows the per-sync-bucket mean center-of-mass distance from the reward "
        "circle center for between-reward segments in Intact Control>Kir "
        "flies in the flat large chamber, used here as a rough measure of segment "
        "skewedness relative to the reward circle center."
    ),
    "export_commands": [
        {
            "label": "Intact Control>Kir",
            "command": 'python analyze.py -v "/media/Synology4/Yang Chen/2025-05-30/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-05-30/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-06-02/c5[12]_*,/media/Synology4/Yang Chen/2025-06-29/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-06-30/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-06-30/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-0[367]/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-07-0[367]/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-0[45]/c5[12]_*,/media/Synology4/Yang Chen/2025-07-05/c6[12]_*,/media/Synology4/Yang Chen/2025-07-11/c5[12]_*,/media/Synology4/Yang Chen/2025-07-11/c6[12]_*,/media/Synology4/Yang Chen/2025-07-26/afternoon/c3[12]_*,/media/Synology4/Yang Chen/2025-07-26/afternoon/c6[12]_*" -f 0-1 --rCC 15 --com-per-segment --com-per-segment-agg mean_magnitude --export-com-sli-bundle exports/com_dist_intact_ctrlKir_flatLgc_inclWall_meanMag_2026-04-13.npz --export-group-label "Intact Control>Kir" --best-worst-sli --best-worst-trn 2 --sli-select-skip-first-sync-buckets 1 --sli-use-training-mean'
        }
    ],
    "plot_commands": [
        {
            "label": "experimental only",
            "command": 'python -m scripts.plot_com_sli_bundles --bundles "exports/com_dist_intact_ctrlKir_flatLgc_inclWall_meanMag_2026-04-13.npz" --out exports/com_dist_intact_ctrlKir_flatLgc_meanMag_2026-04-13.png --metric commag --num-trainings 2',
            "xlabel": None,
            "ylabel": None,
            "output_path": "exports/com_dist_intact_ctrlKir_flatLgc_meanMag_2026-04-13.png"
        },
        {
            "label": "experimental plus yoked controls",
            "command": 'python -m scripts.plot_com_sli_bundles --bundles "exports/com_dist_intact_ctrlKir_flatLgc_inclWall_meanMag_2026-04-13.npz" --out exports/com_dist_intact_ctrlKir_flatLgc_withCtrl_meanMag_2026-04-13.png --metric commag --num-trainings 2 --include-ctrl --show-legend',
            "xlabel": None,
            "ylabel": None,
            "output_path": "exports/com_dist_intact_ctrlKir_flatLgc_withCtrl_meanMag_2026-04-13.png"
        },
        {
            "label": "top 20% vs bottom 50%",
            "command": 'python -m scripts.plot_com_sli_bundles --bundles "exports/com_dist_intact_ctrlKir_flatLgc_inclWall_meanMag_2026-04-13.npz" --out exports/com_dist_intact_ctrlKir_flatLgc_meanMag_top0.2Bot0.5_2026-04-13.png --metric commag --num-trainings 2 --sli-extremes both --top-sli-fraction 0.2 --bottom-sli-fraction 0.5',
            "xlabel": None,
            "ylabel": None,
            "output_path": "exports/com_dist_intact_ctrlKir_flatLgc_meanMag_top0.2Bot0.5_2026-04-13.png"
        }
    ],
}

panel_9


### Panel 9 Data Export Commands


In [ ]:
RUN_PANEL_9_EXPORTS = False
run_stage("Panel 9: export bundles", panel_9["export_commands"], run=RUN_PANEL_9_EXPORTS)


### Panel 9 Plotting Commands


In [ ]:
RUN_PANEL_9_PLOTS = False
run_stage("Panel 9: generate plot", panel_9["plot_commands"], run=RUN_PANEL_9_PLOTS)


## Panel 10

### Panel 10 Plot Description

Panel for **COM distance to reward circle center** comparing **Intact Control>Kir** and **Intact PFN>Kir** flies in the **flat large chamber**.

This panel plots the per-sync-bucket mean of the center-of-mass distance from the reward circle center, computed from between-reward trajectory segments and averaged first within fly and then across flies with confidence intervals. As in panel 9, this metric is used as a rough proxy for the skewedness of the between-reward segment relative to the reward circle center.

The intended interpretation is that the **experimental-minus-yoked** COM-distance difference remains negative for both genotypes, indicating stronger skewedness in yoked controls than in experimental flies overall. At the same time, the **PFN>Kir** flies show a significant positive shift in this exp-minus-yok contrast relative to **Control>Kir**, consistent with the interpretation that, after correcting for genotype-matched yoked controls, experimental PFN>Kir flies have more skewed between-reward segments than Control>Kir flies.

This panel includes one plotting variant:

- Intact Control>Kir vs Intact PFN>Kir, plotted as experimental-minus-yoked curves


In [ ]:
panel_10 = {
    "title": "Panel 10 - COM Distance to Reward Circle Center, Intact Control>Kir vs Intact PFN>Kir",
    "description": (
        "Compares per-sync-bucket center-of-mass distance from the reward "
        "circle center between Intact Control>Kir and Intact "
        "PFN>Kir flies in the flat large chamber, plotted as "
        "experimental-minus-yoked curves to compare skewedness after "
        "correcting for yoked controls."
    ),
    "export_commands": [
        {
            "label": "Intact Control>Kir",
            "command": 'python analyze.py -v "/media/Synology4/Yang Chen/2025-05-30/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-05-30/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-06-02/c5[12]_*,/media/Synology4/Yang Chen/2025-06-29/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-06-30/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-06-30/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-0[367]/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-07-0[367]/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-0[45]/c5[12]_*,/media/Synology4/Yang Chen/2025-07-05/c6[12]_*,/media/Synology4/Yang Chen/2025-07-11/c5[12]_*,/media/Synology4/Yang Chen/2025-07-11/c6[12]_*,/media/Synology4/Yang Chen/2025-07-26/afternoon/c3[12]_*,/media/Synology4/Yang Chen/2025-07-26/afternoon/c6[12]_*" -f 0-1 --rCC 15 --com-per-segment --com-per-segment-agg mean_magnitude --export-com-sli-bundle exports/com_dist_intact_ctrlKir_flatLgc_inclWall_meanMag_2026-04-13.npz --export-group-label "Intact Control>Kir" --best-worst-sli --best-worst-trn 2 --sli-select-skip-first-sync-buckets 1 --sli-use-training-mean'
        },
        {
            "label": "Intact PFN>Kir",
            "command": "python analyze.py -v '/media/Synology4/Yang Chen/2025-05-31/c5[12]_*,/media/Synology4/Yang Chen/2025-05-31/c6[12]_*,/media/Synology4/Yang Chen/2025-05-31/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-06-05/c6[12]_*,/media/Synology4/Yang Chen/2025-06-28/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-06-29/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-0[124]/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-07-0[124]/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-06/c6[12]_*,/media/Synology4/Yang Chen/2025-07-12/night/c32_*,/media/Synology4/Yang Chen/2025-07-21/night/c5[12]_*,/media/Synology4/Yang Chen/2025-07-22/night/c3[12]_*' -f 0-1 --rCC 15 --com-per-segment --com-per-segment-agg mean_magnitude --export-com-sli-bundle exports/com_dist_intact_pfnKir_flatLgc_inclWall_meanMag_2026-04-13.npz --export-group-label \"Intact PFN>Kir\" --best-worst-sli --best-worst-trn 2 --sli-select-skip-first-sync-buckets 1 --sli-use-training-mean"
        }
    ],
    "plot_commands": [
        {
            "label": "Intact Control>Kir vs Intact PFN>Kir",
            "command": 'python -m scripts.plot_com_sli_bundles --bundles "exports/com_dist_intact_ctrlKir_flatLgc_inclWall_meanMag_2026-04-13.npz,exports/com_dist_intact_pfnKir_flatLgc_inclWall_meanMag_2026-04-13.npz" --out exports/com_dist_intact_ctrlKir_vs_intact_pfnKir_inclWall_meanMag_2026-04-21.png --metric commag --turnback-mode exp_minus_ctrl --num-trainings 2',
            "xlabel": None,
            "ylabel": None,
            "output_path": "exports/com_dist_intact_ctrlKir_vs_intact_pfnKir_inclWall_meanMag_2026-04-21.png"
        }
    ],
}

panel_10


### Panel 10 Data Export Commands


In [ ]:
RUN_PANEL_10_EXPORTS = False
run_stage("Panel 10: export bundles", panel_10["export_commands"], run=RUN_PANEL_10_EXPORTS)


### Panel 10 Plotting Commands


In [ ]:
RUN_PANEL_10_PLOTS = False
run_stage("Panel 10: generate plot", panel_10["plot_commands"], run=RUN_PANEL_10_PLOTS)


## Panel 11

### Panel 11 Plot Description

Panel for **SLI, top 20% vs bottom 50%** in **Intact Control>Kir** flies in the **flat HTL chamber**.

For this analysis, the reward preference index (RPI) for a fly is defined as:

$$\mathrm{RPI} = \frac{N_{\mathrm{rwd}} - N_{\mathrm{ctrl}}}{N_{\mathrm{rwd}} + N_{\mathrm{ctrl}}}$$

The spatial learning index (SLI) is then defined as:

$$\mathrm{SLI} = \mathrm{RPI}_{\mathrm{exp}} - \mathrm{RPI}_{\mathrm{yok}}$$

The baseline PI used here is:

$$\mathrm{Baseline\ PI} = \mathrm{RPI}^{\mathrm{pre}}_{\mathrm{exp}} - \mathrm{RPI}^{\mathrm{pre}}_{\mathrm{yok}}$$

This panel shows the per-sync-bucket SLI for the **top 20%** and **bottom 50%** of flies, where percentile groups are defined from the mean SLI in **training 2**, averaged across sync buckets **2-5** after skipping the first sync bucket.

This panel includes one plotting variant:

- top 20% vs bottom 50%


In [ ]:
panel_11 = {
    "title": "Panel 11 - SLI, Top 20% vs Bottom 50%, intact controls, flat HTL",
    "description": (
        "Shows the per-sync-bucket spatial learning index (SLI) for the top "
        "20% and bottom 50% of Intact Control>Kir flies in the flat "
        "HTL chamber, with learner groups defined from the mean SLI in training "
        "2 after skipping the first sync bucket."
    ),
    "export_commands": [
        {
            "label": "Intact Control>Kir (flat HTL)",
            "command": "python analyze.py -v '/media/Synology4/Yang Chen/2024-03-04/c3[12]_*,/media/Synology4/Yang Chen/2024-03-04/c4[12]_*,/media/Synology4/Yang Chen/2024-03-14/c4[12]_*,/media/Synology4/Yang Chen/2024-03-18/c5[12]_*,/media/Synology4/Yang Chen/2024-03-18/c6[12]_*,/media/Synology4/Yang Chen/2024-06-12/c[12]_*' -f 0-9 --rmCC 5 --num-trainings 2 --sli-use-training-mean --sli-select-skip-first-sync-buckets 1 --best-worst-trn 2 --export-com-sli-bundle exports/sli_intact_ctrlKir_flatHTL_2026-04-10.npz --export-group-label \"Intact Control>Kir (flat HTL)\""
        }
    ],
    "plot_commands": [
        {
            "label": "top 20% vs bottom 50%",
            "command": "python -m scripts.plot_com_sli_bundles --bundles \"/home/tracking/code/YL/nvsl-analysis/exports/sli_intact_ctrlKir_flatHTL_2026-04-10.npz\" --out exports/sli_intact_ctrlKir_flatHTL_top0.2Bot0.5_2026-04-10.png --metric sli --num-trainings 2 --sli-extremes both --top-sli-fraction 0.2 --bottom-sli-fraction 0.5",
            "xlabel": None,
            "ylabel": None,
            "output_path": "exports/sli_intact_ctrlKir_flatHTL_top0.2Bot0.5_2026-04-10.png"
        }
    ],
}

panel_11

### Panel 11 Data Export Commands


In [ ]:
RUN_PANEL_11_EXPORTS = False
run_stage("Panel 11: export bundles", panel_11["export_commands"], run=RUN_PANEL_11_EXPORTS)


### Panel 11 Plotting Commands


In [ ]:
RUN_PANEL_11_PLOTS = False
run_stage("Panel 11: generate plot", panel_11["plot_commands"], run=RUN_PANEL_11_PLOTS)


## Panel 12

### Panel 12 Plot Description

Panel for **post-session SLI, top 20% vs bottom 50%** in **Intact Control>Kir** flies in the **flat HTL chamber**.

For this analysis, the reward preference index (RPI) for a fly is defined as:

$$\mathrm{RPI} = \frac{N_{\mathrm{rwd}} - N_{\mathrm{ctrl}}}{N_{\mathrm{rwd}} + N_{\mathrm{ctrl}}}$$

The spatial learning index (SLI) is then defined as:

$$\mathrm{SLI} = \mathrm{RPI}_{\mathrm{exp}} - \mathrm{RPI}_{\mathrm{yok}}$$

The baseline PI used here is:

$$\mathrm{Baseline\ PI} = \mathrm{RPI}^{\mathrm{pre}}_{\mathrm{exp}} - \mathrm{RPI}^{\mathrm{pre}}_{\mathrm{yok}}$$

This panel uses the same learner split as panel 10, defining the **top 20%** and **bottom 50%** from the mean SLI in **training 2**, averaged across sync buckets **2-5** after skipping the first sync bucket. The plotted trace is the **post-session** reward PI difference over the fifteen-minute period after training, when flies can still enter the reward circle but no rewards are delivered.

The intended interpretation is that the **bottom 50%** group drops to nearly zero immediately in the post-session, whereas the **top 20%** group shows a more gradual decay, suggesting more persistent reward-circle preference after reward delivery stops.

This is a one-shot `analyze.py` command that generates the post-session figure directly, so there is no separate bundle export stage for this panel.


In [ ]:
panel_12 = {
    "title": "Panel 12 - Post-session SLI, Top 20% vs Bottom 50%, Intact Control>Kir flies in the flat HTL chamber",
    "description": (
        "Shows post-session reward PI difference traces for the top 20% and "
        "bottom 50% of Intact Control>Kir flies in the flat HTL "
        "chamber, using the same training-2 SLI-based learner split as panel "
        "10 and emphasizing the faster post-reward decay in the bottom 50%."
    ),
    "export_commands": [],
    "plot_commands": [
        {
            "label": "post-session, top 20% vs bottom 50%",
            "command": "python analyze.py -v \"/media/Synology4/Yang Chen/2024-03-04/c3[12]_*,/media/Synology4/Yang Chen/2024-03-04/c4[12]_*,/media/Synology4/Yang Chen/2024-03-14/c4[12]_*,/media/Synology4/Yang Chen/2024-03-18/c5[12]_*,/media/Synology4/Yang Chen/2024-03-18/c6[12]_*,/media/Synology4/Yang Chen/2024-06-12/c[12]_*\" -f 0-9 --rmCC 5 --best-worst-extreme both --sli-use-training-mean --best-worst-trn 2 --sli-select-skip-first-sync-buckets 1 --top-sli-fraction 0.2 --bottom-sli-fraction 0.5 --num-trainings 2",
            "plot_rewards_xlabel": None,
            "plot_rewards_ylabel": None,
            "rename_outputs": [
                {
                    "from": "imgs/reward_pi_post_diff__3_min_buckets_bottom50_top20.png",
                    "to": "imgs/rpi_post_diff__3_min_buckets_bottom50_top20_intact_ctrlKir_flatHTL_2026-04-13.png"
                }
            ],
            "output_path": "imgs/rpi_post_diff__3_min_buckets_bottom50_top20_intact_ctrlKir_flatHTL_2026-04-13.png"
        }
    ],
}

panel_12

### Panel 12 Data Export Commands


In [ ]:
RUN_PANEL_12_EXPORTS = False
run_stage("Panel 12: export bundles", panel_12["export_commands"], run=RUN_PANEL_12_EXPORTS)


### Panel 12 Plotting Commands


In [ ]:
RUN_PANEL_12_PLOTS = False
run_stage("Panel 12: generate plot", panel_12["plot_commands"], run=RUN_PANEL_12_PLOTS)


## Panel 13

### Panel 13 Plot Description

Panel for **SLI versus first-10 rewards rate** in **Intact Control>Kir** flies across the two **HTL chamber variants**: **flat** and **agarose**.

For this analysis, the reward preference index (RPI) for a fly is defined as:

$$\mathrm{RPI} = \frac{N_{\mathrm{rwd}} - N_{\mathrm{ctrl}}}{N_{\mathrm{rwd}} + N_{\mathrm{ctrl}}}$$

The spatial learning index (SLI) is then defined as:

$$\mathrm{SLI} = \mathrm{RPI}_{\mathrm{exp}} - \mathrm{RPI}_{\mathrm{yok}}$$

The baseline PI used here is:

$$\mathrm{Baseline\ PI} = \mathrm{RPI}^{\mathrm{pre}}_{\mathrm{exp}} - \mathrm{RPI}^{\mathrm{pre}}_{\mathrm{yok}}$$

This panel shows two related correlation variants between SLI and the reward rate computed from the first **10 calculated rewards** in **training 1**, repeated for both HTL chamber variants:

- SLI from the first sync bucket of training 1
- SLI averaged across training 1 sync buckets 2-5

Each variant is run for:

- flat HTL chamber
- agarose HTL chamber

These are one-shot `analyze.py` commands that generate the scatter plots directly, so there is no separate bundle export stage for this panel.


In [ ]:
panel_13 = {
    "title": "Panel 13 - SLI versus first-10 rewards rate, intact controls, HTL chambers",
    "description": (
        "Shows two correlation views between spatial learning index (SLI) and "
        "the reward rate for the first 10 calculated rewards in Intact "
        "Control>Kir flies across the flat and agarose HTL chamber "
        "variants: one using only the first sync bucket of training 1, and "
        "one using the mean SLI across training 1 sync buckets 2-5."
    ),
    "export_commands": [],
    "plot_commands": [
        {
            "label": "T1 sync bucket 1",
            "command": "python analyze.py -v '/media/Synology4/Yang Chen/2024-03-04/c3[12]_*,/media/Synology4/Yang Chen/2024-03-04/c4[12]_*,/media/Synology4/Yang Chen/2024-03-14/c4[12]_*,/media/Synology4/Yang Chen/2024-03-18/c5[12]_*,/media/Synology4/Yang Chen/2024-03-18/c6[12]_*,/media/Synology4/Yang Chen/2024-06-12/c[12]_*' -f 0-9 --rmCC 5 --first-n-reward-diagnostics --first-n-reward-diagnostics-reward-event-type calc --first-n-reward-diagnostics-pi-threshold 10 --first-n-reward-diagnostics-sli-group all --first-n-reward-diagnostics-trainings 1 --first-n-reward-diagnostics-n 10 --first-n-reward-diagnostics-x-by selected_reward_rate_to_nth_per_min --first-n-reward-diagnostics-y-by sli --first-n-reward-diagnostics-color-by none --best-worst-sli --best-worst-trn 1 --sli-use-training-mean --sli-select-keep-first-sync-buckets 1 --first-n-reward-diagnostics-csv exports/sliT1SB1_vs_rwdsFirst10_intact_ctrlKir_flatHTL_2026-04-10.csv --first-n-reward-diagnostics-plot-out exports/sliT1SB1_vs_rwdsFirst10_intact_ctrlKir_flatHTL_2026-04-10.png",
            "first_n_reward_diagnostics_xlabel": None,
            "first_n_reward_diagnostics_ylabel": None,
            "output_path": "exports/sliT1SB1_vs_rwdsFirst10_intact_ctrlKir_flatHTL_2026-04-10.png"
        },
        {
            "label": "flat HTL chamber, T1 sync buckets 2-5 mean",
            "command": "python analyze.py -v '/media/Synology4/Yang Chen/2024-03-04/c3[12]_*,/media/Synology4/Yang Chen/2024-03-04/c4[12]_*,/media/Synology4/Yang Chen/2024-03-14/c4[12]_*,/media/Synology4/Yang Chen/2024-03-18/c5[12]_*,/media/Synology4/Yang Chen/2024-03-18/c6[12]_*,/media/Synology4/Yang Chen/2024-06-12/c[12]_*' -f 0-9 --rmCC 5 --first-n-reward-diagnostics --first-n-reward-diagnostics-reward-event-type calc --first-n-reward-diagnostics-pi-threshold 10 --first-n-reward-diagnostics-sli-group all --first-n-reward-diagnostics-trainings 1 --first-n-reward-diagnostics-n 10 --first-n-reward-diagnostics-x-by selected_reward_rate_to_nth_per_min --first-n-reward-diagnostics-y-by sli --first-n-reward-diagnostics-color-by none --best-worst-sli --best-worst-trn 1 --sli-use-training-mean --skip-first-sync-buckets 1 --sli-select-skip-first-sync-buckets 1 --first-n-reward-diagnostics-csv exports/sliT1SB2SB5_vs_rwdsFirst10_intact_ctrlKir_flatHTL_2026-04-10.csv --first-n-reward-diagnostics-plot-out exports/sliT1SB2SB5_vs_rwdsFirst10_intact_ctrlKir_flatHTL_2026-04-10.png",
            "first_n_reward_diagnostics_xlabel": None,
            "first_n_reward_diagnostics_ylabel": None,
            "output_path": "exports/sliT1SB2SB5_vs_rwdsFirst10_intact_ctrlKir_flatHTL_2026-04-10.png"
        },
        {
            "label": "agarose HTL chamber, T1 sync bucket 1",
            "command": "python analyze.py -v '/media/Synology4/Yang Chen/2024-03-1[34]/c5[12]_*,/media/Synology4/Yang Chen/2024-03-1[347]/c6[12]_*,/media/Synology4/Yang Chen/2024-03-18/c[12]_*' -f 0-9 --rmCC 5 --first-n-reward-diagnostics --first-n-reward-diagnostics-reward-event-type calc --first-n-reward-diagnostics-pi-threshold 10 --first-n-reward-diagnostics-sli-group all --first-n-reward-diagnostics-trainings 1 --first-n-reward-diagnostics-n 10 --first-n-reward-diagnostics-x-by selected_reward_rate_to_nth_per_min --first-n-reward-diagnostics-y-by sli --first-n-reward-diagnostics-color-by none --best-worst-sli --best-worst-trn 1 --sli-use-training-mean --sli-select-keep-first-sync-buckets 1 --first-n-reward-diagnostics-csv exports/sliT1SB1_vs_rwdsFirst10_intact_ctrlKir_agaroseHTL_2026-04-20.csv --first-n-reward-diagnostics-plot-out exports/sliT1SB1_vs_rwdsFirst10_intact_ctrlKir_agaroseHTL_2026-04-20.png",
            "first_n_reward_diagnostics_xlabel": None,
            "first_n_reward_diagnostics_ylabel": None,
            "output_path": "exports/sliT1SB1_vs_rwdsFirst10_intact_ctrlKir_agaroseHTL_2026-04-20.png"
        },
        {
            "label": "agarose HTL chamber, T1 sync buckets 2-5 mean",
            "command": "python analyze.py -v '/media/Synology4/Yang Chen/2024-03-1[34]/c5[12]_*,/media/Synology4/Yang Chen/2024-03-1[347]/c6[12]_*,/media/Synology4/Yang Chen/2024-03-18/c[12]_*' -f 0-9 --rmCC 5 --first-n-reward-diagnostics --first-n-reward-diagnostics-reward-event-type calc --first-n-reward-diagnostics-pi-threshold 10 --first-n-reward-diagnostics-sli-group all --first-n-reward-diagnostics-trainings 1 --first-n-reward-diagnostics-n 10 --first-n-reward-diagnostics-x-by selected_reward_rate_to_nth_per_min --first-n-reward-diagnostics-y-by sli --first-n-reward-diagnostics-color-by none --best-worst-sli --best-worst-trn 1 --sli-use-training-mean --skip-first-sync-buckets 1 --sli-select-skip-first-sync-buckets 1 --first-n-reward-diagnostics-csv exports/sliT1SB2SB5_vs_rwdsFirst10_intact_ctrlKir_agaroseHTL_2026-04-20.csv --first-n-reward-diagnostics-plot-out exports/sliT1SB2SB5_vs_rwdsFirst10_intact_ctrlKir_agaroseHTL_2026-04-20.png",
            "first_n_reward_diagnostics_xlabel": None,
            "first_n_reward_diagnostics_ylabel": None,
            "output_path": "exports/sliT1SB2SB5_vs_rwdsFirst10_intact_ctrlKir_agaroseHTL_2026-04-20.png"
        }
    ],
}

panel_13

### Panel 13 Data Export Commands


In [ ]:
RUN_PANEL_13_EXPORTS = False
run_stage("Panel 13: export bundles", panel_13["export_commands"], run=RUN_PANEL_13_EXPORTS)


### Panel 13 Plotting Commands


In [ ]:
RUN_PANEL_13_PLOTS = False
run_stage("Panel 13: generate plot", panel_13["plot_commands"], run=RUN_PANEL_13_PLOTS)


## Panel 14

### Panel 14 Plot Description

Panel for **SLI, top 20% vs bottom 50%** in **Intact Control>Kir** flies in the **flat large chamber**.

For this analysis, the reward preference index (RPI) for a fly is defined as:

$$\mathrm{RPI} = \frac{N_{\mathrm{rwd}} - N_{\mathrm{ctrl}}}{N_{\mathrm{rwd}} + N_{\mathrm{ctrl}}}$$

The spatial learning index (SLI) is then defined as:

$$\mathrm{SLI} = \mathrm{RPI}_{\mathrm{exp}} - \mathrm{RPI}_{\mathrm{yok}}$$

The baseline PI used here is:

$$\mathrm{Baseline\ PI} = \mathrm{RPI}^{\mathrm{pre}}_{\mathrm{exp}} - \mathrm{RPI}^{\mathrm{pre}}_{\mathrm{yok}}$$

This panel shows the per-sync-bucket SLI for the **top 20%** and **bottom 50%** of flies, where percentile groups are defined from the mean SLI in **training 2**, averaged across sync buckets **2-5** after skipping the first sync bucket.

This panel includes one plotting variant:

- top 20% vs bottom 50%


In [ ]:
panel_14 = {
    "title": "Panel 14 - SLI, Top 20% vs Bottom 50%, intact controls, flat large",
    "description": (
        "Shows the per-sync-bucket spatial learning index (SLI) for the top "
        "20% and bottom 50% of Intact Control>Kir flies in the flat "
        "large chamber, with learner groups defined from the mean SLI in "
        "training 2 across sync buckets 2-5 after skipping the first sync "
        "bucket."
    ),
    "export_commands": [
        {
            "label": "Intact Control>Kir (flat large)",
            "command": "python analyze.py -v \"/media/Synology4/Yang Chen/2025-05-30/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-05-30/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-06-02/c5[12]_*,/media/Synology4/Yang Chen/2025-06-29/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-06-30/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-06-30/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-0[367]/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-07-0[367]/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-0[45]/c5[12]_*,/media/Synology4/Yang Chen/2025-07-05/c6[12]_*,/media/Synology4/Yang Chen/2025-07-11/c5[12]_*,/media/Synology4/Yang Chen/2025-07-11/c6[12]_*,/media/Synology4/Yang Chen/2025-07-26/afternoon/c3[12]_*,/media/Synology4/Yang Chen/2025-07-26/afternoon/c6[12]_*\" -f 0-1 --rCC 15 --top-sli-fraction 0.2 --bottom-sli-fraction 0.5 --num-trainings 2 --sli-use-training-mean --sli-select-skip-first-sync-buckets 1 --export-com-sli-bundle exports/sli_intact_ctrlKir_flatLgc_2026-04-10.npz --best-worst-trn 2 --export-group-label \"Intact Control>Kir\""
        }
    ],
    "plot_commands": [
        {
            "label": "top 20% vs bottom 50%",
            "command": "python -m scripts.plot_com_sli_bundles --bundles \"/home/tracking/code/YL/nvsl-analysis/exports/sli_intact_ctrlKir_flatLgc_2026-04-10.npz\" --out exports/sli_intact_ctrlKir_flatLgc_top0.2Bot0.5_2026-04-10.png --metric sli --num-trainings 2 --sli-extremes both --top-sli-fraction 0.2 --bottom-sli-fraction 0.5",
            "xlabel": None,
            "ylabel": None,
            "output_path": "exports/sli_intact_ctrlKir_flatLgc_top0.2Bot0.5_2026-04-10.png"
        }
    ],
}

panel_14

### Panel 14 Data Export Commands


In [ ]:
RUN_PANEL_14_EXPORTS = False
run_stage("Panel 14: export bundles", panel_14["export_commands"], run=RUN_PANEL_14_EXPORTS)


### Panel 14 Plotting Commands


In [ ]:
RUN_PANEL_14_PLOTS = False
run_stage("Panel 14: generate plot", panel_14["plot_commands"], run=RUN_PANEL_14_PLOTS)


## Panel 15

### Panel 15 Plot Description

Panel for **agarose avoidance ratio, large chamber, flat versus agarose** across three fly groups.

This panel compares **Intact Control>Kir**, **AR Control>Kir**, and **Intact PFN>Kir** flies over the first two trainings in the **large chamber**, contrasting real agarose wells against matched flat-chamber controls.

The agarose avoidance ratio is defined from episodes that begin when a fly enters a padded approach region around an agarose well and are classified by whether the fly later enters the agarose well itself. The plotted metric summarizes the fraction of these approach episodes that remain avoidances rather than contacts.

The intended interpretation is that AR Control>Kir flies show nearly the same avoidance ratio in flat and agarose chambers, whereas the two intact groups show stronger avoidance in the agarose chamber than in the flat chamber's virtual comparison.

This panel includes three plotting variants:

- Intact Control>Kir
- AR Control>Kir
- Intact PFN>Kir


In [ ]:
panel_15 = {
    "title": "Panel 15 - Agarose avoidance ratio, large chamber, flat versus agarose",
    "description": (
        "Compares agarose avoidance ratio across the first two trainings in the "
        "large chamber for Intact Control>Kir, AR Control>Kir, "
        "and Intact PFN>Kir flies, contrasting real agarose wells with flat-"
        "chamber virtual agarose controls."
    ),
    "export_commands": [
        {
            "label": "Intact Control>Kir (flat large)",
            "command": "python analyze.py -v \"/media/Synology4/Yang Chen/2025-05-30/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-05-30/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-06-02/c5[12]_*,/media/Synology4/Yang Chen/2025-06-29/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-06-30/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-06-30/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-0[367]/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-07-0[367]/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-0[45]/c5[12]_*,/media/Synology4/Yang Chen/2025-07-05/c6[12]_*,/media/Synology4/Yang Chen/2025-07-11/c5[12]_*,/media/Synology4/Yang Chen/2025-07-11/c6[12]_*,/media/Synology4/Yang Chen/2025-07-26/afternoon/c3[12]_*,/media/Synology4/Yang Chen/2025-07-26/afternoon/c6[12]_*\" -f 0-1 --rCC 15 --agarose-dual-circle --agarose-outer-delta-mm 1 --export-agarose-sli-bundle \"exports/agarose_avoid_1mm_intact_ctrlKir_flatLgc_2026-04-10.npz\" --agarose-sli-include-pre --export-group-label \"Intact Control>Kir\""
        },
        {
            "label": "Intact Control>Kir (agarose large)",
            "command": "python analyze.py -v \"/media/Synology4/Yang Chen/2025-06-28/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-10/night/c5[12]_*,/media/Synology4/Yang Chen/2025-07-10/night/c6[12]_*,/media/Synology4/Yang Chen/2025-07-1[26]/c6[12]_*,/media/Synology4/Yang Chen/2025-07-1[26]/c3[12]_*,/media/Synology4/Yang Chen/2025-07-1[36]/afternoon/c3[12]_*,/media/Synology4/Yang Chen/2025-07-1[36]/afternoon/c4[12]_*,/media/Synology4/Yang Chen/2025-07-1[6]/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-07-1[6]/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-16/c4[12]_*,/media/Synology4/Yang Chen/2025-07-16/c5[12]_*,/media/Synology4/Yang Chen/2025-07-2[25]/night/c5[12]_*,/media/Synology4/Yang Chen/2025-07-2[5]/night/c3[12]_*,/media/Synology4/Yang Chen/2025-07-2[5]/night/c4[12]_*,/media/Synology4/Yang Chen/2025-07-2[5]/night/c6[12]_*\" -f 0-1 --rCC 15 --agarose-dual-circle --agarose-outer-delta-mm 1 --export-agarose-sli-bundle \"exports/agarose_avoid_1mm_intact_ctrlKir_agaroseLgc_2026-04-10.npz\" --agarose-sli-include-pre --export-group-label \"Intact Control>Kir (agarose large)\""
        },
        {
            "label": "AR Control>Kir (flat large)",
            "command": "python analyze.py -v \"/media/Synology4/Yang Chen/2025-07-17/c4[12]_*,/media/Synology4/Yang Chen/2025-07-17/c5[12]_*,/media/Synology4/Yang Chen/2025-07-17/c6[12]_*,/media/Synology4/Yang Chen/2025-07-18/night/c5[12]_*,/media/Synology4/Yang Chen/2025-07-18/night/c6[12]_*,/media/Synology4/Yang Chen/2025-07-2[09]/night/c3[12]_*,/media/Synology4/Yang Chen/2025-07-2[01]/night/c4[12]_*,/media/Synology4/Yang Chen/2025-07-21/night/c6[12]_*,/media/Synology4/Yang Chen/2025-07-27/c6[12]_*,/media/Synology4/Yang Chen/2025-07-27/c5[12]_*,/media/Synology4/Yang Chen/2025-07-27/afternoon/c3[12]_*,/media/Synology4/Yang Chen/2025-07-27/afternoon/c4[12]_*,/media/Synology4/Yang Chen/2025-07-2[79]/night/c4[12]_*,/media/Synology4/Yang Chen/2025-07-30/night/c3[12]_*,/media/Synology4/Yang Chen/2025-07-30/night/c4[12]_*,/media/Synology4/Yang Chen/2025-08-02/night/c3[12]_*\" -f 0-1 --rCC 15 --agarose-dual-circle --agarose-outer-delta-mm 1 --export-agarose-sli-bundle \"exports/agarose_avoid_1mm_ar_ctrlKir_flatLgc_2026-04-10.npz\" --agarose-sli-include-pre --export-group-label \"AR Control>Kir\""
        },
        {
            "label": "AR Control>Kir (agarose large)",
            "command": "python analyze.py -v \"/media/Synology4/Yang Chen/2025-07-17/night/c5[12]_*,/media/Synology4/Yang Chen/2025-07-27/c3[12]_*,/media/Synology4/Yang Chen/2025-07-27/c4[12]_*,/media/Synology4/Yang Chen/2025-07-27/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-07-27/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-2[789]/night/c5[12]_*,/media/Synology4/Yang Chen/2025-07-2[789]/night/c6[12]_*,/media/Synology4/Yang Chen/2025-07-28/night/c3[12]_*,/media/Synology4/Yang Chen/2025-07-28/night/c4[12]_*,/media/Synology4/Yang Chen/2025-07-30/night/c5[12]_*,/media/Synology4/Yang Chen/2025-07-30/night/c6[12]_*,/media/Synology4/Yang Chen/2025-08-01/night/c3[12]_*,/media/Synology4/Yang Chen/2025-08-01/night/c4[12]_*,/media/Synology4/Yang Chen/2025-08-02/night/c5[12]_*\" -f 0-1 --rCC 15 --agarose-dual-circle --agarose-outer-delta-mm 1 --export-agarose-sli-bundle \"exports/agarose_avoid_1mm_ar_ctrlKir_agaroseLgc_2026-04-10.npz\" --agarose-sli-include-pre --export-group-label \"AR Control>Kir (agarose large)\""
        },
        {
            "label": "Intact PFN>Kir (flat large)",
            "command": "python analyze.py -v \"/media/Synology4/Yang Chen/2025-05-31/c5[12]_*,/media/Synology4/Yang Chen/2025-05-31/c6[12]_*,/media/Synology4/Yang Chen/2025-05-31/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-06-05/c6[12]_*,/media/Synology4/Yang Chen/2025-06-28/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-06-29/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-0[124]/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-07-0[124]/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-06/c6[12]_*,/media/Synology4/Yang Chen/2025-07-12/night/c32_*,/media/Synology4/Yang Chen/2025-07-21/night/c5[12]_*,/media/Synology4/Yang Chen/2025-07-22/night/c3[12]_*\" -f 0-1 --rCC 15 --agarose-dual-circle --agarose-outer-delta-mm 1 --export-agarose-sli-bundle \"exports/agarose_avoid_1mm_intact_pfnKir_flatLgc_2026-04-10.npz\" --agarose-sli-include-pre --export-group-label \"Intact PFN>Kir\""
        },
        {
            "label": "Intact PFN>Kir (agarose large)",
            "command": "python analyze.py -v \"/media/Synology4/Yang Chen/2025-07-0[8]/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-07-0[8]/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-10/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-07-10/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-1[126]/night/c5[12]_*,/media/Synology4/Yang Chen/2025-07-1[129]/night/c6[12]_*,/media/Synology4/Yang Chen/2025-07-1[2]/night/c4[12]_*,/media/Synology4/Yang Chen/2025-07-1[69]/night/c3[12]_*,/media/Synology4/Yang Chen/2025-07-15/c3[12]_*,/media/Synology4/Yang Chen/2025-07-15/c4[12]_*,/media/Synology4/Yang Chen/2025-07-15/c5[12]_*,/media/Synology4/Yang Chen/2025-07-15/c6[12]_*,/media/Synology4/Yang Chen/2025-07-22/night/c6[12]_*\" -f 0-1 --rCC 15 --agarose-dual-circle --agarose-outer-delta-mm 1 --export-agarose-sli-bundle \"exports/agarose_avoid_1mm_intact_pfnKir_agaroseLgc_2026-04-10.npz\" --agarose-sli-include-pre --export-group-label \"Intact PFN>Kir (agarose large)\""
        }
    ],
    "plot_commands": [
        {
            "label": "Intact Control>Kir",
            "command": "python -m scripts.plot_com_sli_bundles --bundles \"/home/tracking/code/YL/nvsl-analysis/exports/agarose_avoid_1mm_intact_ctrlKir_flatLgc_2026-04-10.npz,/home/tracking/code/YL/nvsl-analysis/exports/agarose_avoid_1mm_intact_ctrlKir_agaroseLgc_2026-04-10.npz\" --metric agarose --turnback-mode exp --include-pre --out \"exports/agarose_avoid_1mm_intact_ctrlKir_2026-04-10.png\"",
            "xlabel": None,
            "ylabel": None,
            "output_path": "exports/agarose_avoid_1mm_intact_ctrlKir_2026-04-10.png"
        },
        {
            "label": "AR Control>Kir",
            "command": "python -m scripts.plot_com_sli_bundles --bundles \"/home/tracking/code/YL/nvsl-analysis/exports/agarose_avoid_1mm_ar_ctrlKir_flatLgc_2026-04-10.npz,/home/tracking/code/YL/nvsl-analysis/exports/agarose_avoid_1mm_ar_ctrlKir_agaroseLgc_2026-04-10.npz\" --metric agarose --turnback-mode exp --include-pre --out \"exports/agarose_avoid_1mm_ar_ctrlKir_2026-04-10.png\"",
            "xlabel": None,
            "ylabel": None,
            "output_path": "exports/agarose_avoid_1mm_ar_ctrlKir_2026-04-10.png"
        },
        {
            "label": "Intact PFN>Kir",
            "command": "python -m scripts.plot_com_sli_bundles --bundles \"/home/tracking/code/YL/nvsl-analysis/exports/agarose_avoid_1mm_intact_pfnKir_flatLgc_2026-04-10.npz,/home/tracking/code/YL/nvsl-analysis/exports/agarose_avoid_1mm_intact_pfnKir_agaroseLgc_2026-04-10.npz\" --metric agarose --turnback-mode exp --include-pre --out \"exports/agarose_avoid_1mm_intact_pfnKir_2026-04-10.png\"",
            "xlabel": None,
            "ylabel": None,
            "output_path": "exports/agarose_avoid_1mm_intact_pfnKir_2026-04-10.png"
        }
    ],
}

panel_15

### Panel 15 Data Export Commands


In [ ]:
RUN_PANEL_15_EXPORTS = False
run_stage("Panel 15: export bundles", panel_15["export_commands"], run=RUN_PANEL_15_EXPORTS)


### Panel 15 Plotting Commands


In [ ]:
RUN_PANEL_15_PLOTS = False
run_stage("Panel 15: generate plot", panel_15["plot_commands"], run=RUN_PANEL_15_PLOTS)


## Panel 16

### Panel 16 Plot Description

Panel for **sharp turn probability** in **training 2** in the **flat HTL chamber**, comparing **Intact Control>Kir** and **AR Control>Kir** flies.

This analysis uses the **circular** boundary-contact formulation around the reward circle, with outside-event radii at **2, 3, 4, and 5 mm** from the reward circle center. At each radius, the plotted quantity is the fraction of outside-circle exit events that are classified as sharp turns.

Here, a sharp turn follows the `findTurns` logic in `src/analysis/boundary_contact.pyx`: a candidate event must remain short in duration (default threshold **0.75 s**) while accumulating sufficient reorientation (default minimum **90 degrees** of velocity-angle change), excluding wall-contact frames and frames below the minimum turn-speed threshold.

The intended biological interpretation is that Intact Control>Kir flies show a higher sharp-turn probability than AR Control>Kir flies beginning at approximately **3 mm** from the reward center, consistent with the idea that intact flies may use scent deposited near the reward to guide more effective returns.

This is a one-shot `analyze.py` command that generates several turn-probability figures directly. The relevant panel output here is the **training 2 end, all-turns, across-groups** plot.


In [ ]:
panel_16 = {
    "title": "Panel 16 - Sharp turn probability, flat HTL chamber, intact vs AR Control>Kir",
    "description": (
        "Compares circular-boundary sharp turn probability during training 2 in "
        "flat HTL chambers between Intact and AR "
        "Control>Kir flies, emphasizing the higher Intact Control>Kir sharp-turn "
        "probability from roughly 3 mm outward from the reward center."
    ),
    "export_commands": [],
    "plot_commands": [
        {
            "label": "training 2 end, all sharp turns, across groups",
            "command": "python analyze.py -v \"/media/Synology4/Yang Chen/2024-03-04/c3[12]_*,/media/Synology4/Yang Chen/2024-03-04/c4[12]_*,/media/Synology4/Yang Chen/2024-03-14/c4[12]_*,/media/Synology4/Yang Chen/2024-03-18/c5[12]_*,/media/Synology4/Yang Chen/2024-03-18/c6[12]_*,/media/Synology4/Yang Chen/2024-06-12/c[12]_*|/media/Synology4/Yang Chen/2024-03-1[19]/c3[12]_*,/media/Synology4/Yang Chen/2024-03-11/c4[12]_*,/media/Synology4/Yang Chen/2024-03-11/c5[12]_*,/media/Synology4/Yang Chen/2024-03-11/c6[12]_*,/media/Synology4/Yang Chen/2024-06-12/c2[12]_*,/media/Synology4/Yang Chen/2024-06-12/c3[12]_*,/media/Synology4/Yang Chen/2024-06-12/c4[12]_*\" -f 0-9 --rmCC 5 --gl \"Ctrl|Antenna removed\" --turn boundary --end --minNumLT 20 --outside-circle-radii 2,3,4,5 --turn boundary --contact-geometry circular",
            "turn_prob_dist_xlabel": None,
            "turn_prob_dist_ylabel": None,
            "rename_outputs": [
                {
                    "from": "imgs/turn_probability_t2_end_all_exp_across_groups.png",
                    "to": "imgs/turn_probability_t2_end_all_exp_across_groups_panel15_intact_vs_ar_ctrlKir_2026-04-17.png"
                }
            ],
            "output_path": "imgs/turn_probability_t2_end_all_exp_across_groups_panel15_intact_vs_ar_ctrlKir_2026-04-17.png"
        }
    ],
}

panel_16

### Panel 16 Data Export Commands


In [ ]:
RUN_PANEL_16_EXPORTS = False
run_stage("Panel 16: export bundles", panel_16["export_commands"], run=RUN_PANEL_16_EXPORTS)


### Panel 16 Plotting Commands


In [ ]:
RUN_PANEL_16_PLOTS = False
run_stage("Panel 16: generate plot", panel_16["plot_commands"], run=RUN_PANEL_16_PLOTS)


## Panel 17

### Panel 17 Plot Description

Panel for **sharp turn probability** in **training 2** in the **flat HTL chamber**, comparing **Control** and **hind tarsi removed + genitalia glued** flies.

This analysis uses the **circular** boundary-contact formulation around the reward circle, with outside-event radii at **2, 3, 4, and 5 mm** from the reward circle center. At each radius, the plotted quantity is the fraction of outside-circle exit events that are classified as sharp turns.

Here, a sharp turn follows the `findTurns` logic in `src/analysis/boundary_contact.pyx`: a candidate event must remain short in duration (default threshold **0.75 s**) while accumulating sufficient reorientation (default minimum **90 degrees** of velocity-angle change), excluding wall-contact frames and frames below the minimum turn-speed threshold.

This panel mirrors the analysis setup used in **Panel 15**, but swaps in the Control versus hind-tarsi-removed-plus-genitalia-glued cohort definition and adds `--memory-saver` because the sample size is too large for the full run to complete otherwise.

This is a one-shot `analyze.py` command that generates several turn-probability figures directly. The relevant panel output here is the **training 2 end, all-turns, across-groups** plot.


In [ ]:
panel_17 = {
    "title": "Panel 17 - Sharp turn probability, flat HTL chamber, Control vs hind tarsi removed + genitalia glued",
    "description": (
        "Compares circular-boundary sharp turn probability during training 2 in "
        "flat HTL chambers between Control flies and hind-tarsi-removed-plus-"
        "genitalia-glued flies, using the same outside-event radii as panel 15 "
        "with memory-saver mode enabled for the larger cohort."
    ),
    "export_commands": [],
    "plot_commands": [
        {
            "label": "training 2 end, all sharp turns, across groups",
            "command": "python analyze.py -v \"/media/Synology4/Yang Chen/2024-07-1[45]/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2024-07-1[45]/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2024-07-1[45]/afternoon/c3[12]_*,/media/Synology4/Yang Chen/2024-07-1[45]/afternoon/c4[12]_*,/media/Synology4/Yang Chen/2024-09-04/afternoon/c4[12]_*,/media/Synology4/Yang Chen/2024-09-04/afternoon/c32_*|/media/Synology4/Yang Chen/2024-06-03/c2[12]_*,/media/Synology4/Yang Chen/2024-06-03/c[12]_*,/media/Synology4/Yang Chen/2024-06-03/c3[12]_*,/media/Synology4/Yang Chen/2024-06-0[45]/afternoon/c2[12]_*,/media/Synology4/Yang Chen/2024-06-05/afternoon/c[12]_*,/media/Synology4/Yang Chen/2024-06-0[89]/c3[12]_*,/media/Synology4/Yang Chen/2024-06-09/c4[12]_*,/media/Synology4/Yang Chen/2024-06-10/c3[12]_*,/media/Synology4/Yang Chen/2024-06-10/c4[12]_*,/media/Synology4/Yang Chen/2024-06-1[13]/afternoon/c4[12]_*,/media/Synology4/Yang Chen/2024-08-29/afternoon/c3[12]_*,/media/Synology4/Yang Chen/2024-08-29/afternoon/c4[12]_*,/media/Synology4/Yang Chen/2024-09-04/afternoon/c[12]_*,/media/Synology4/Yang Chen/2024-09-04/afternoon/c2[12]_*,/media/Synology4/Yang Chen/2024-09-04/afternoon/c31_*\" -f 0-9 --rmCC 5 --gl \"Ctrl|Hind tarsi removed + genitalia glued\" --turn boundary --end --minNumLT 20 --outside-circle-radii 2,3,4,5 --turn boundary --contact-geometry circular --memory-saver",
            "turn_prob_dist_xlabel": None,
            "turn_prob_dist_ylabel": None,
            "rename_outputs": [
                {
                    "from": "imgs/turn_probability_t2_end_all_exp_across_groups.png",
                    "to": "imgs/turn_probability_t2_end_all_exp_across_groups_panel16_ctrl_vs_ht_removed_gen_glued_2026-04-17.png"
                }
            ],
            "output_path": "imgs/turn_probability_t2_end_all_exp_across_groups_panel16_ctrl_vs_ht_removed_gen_glued_2026-04-17.png"
        }
    ],
}

panel_17

### Panel 17 Data Export Commands


In [ ]:
RUN_PANEL_17_EXPORTS = False
run_stage("Panel 17: export bundles", panel_17["export_commands"], run=RUN_PANEL_17_EXPORTS)


### Panel 17 Plotting Commands


In [ ]:
RUN_PANEL_17_PLOTS = False
run_stage("Panel 17: generate plot", panel_17["plot_commands"], run=RUN_PANEL_17_PLOTS)


## Panel 18

### Panel 18 Plot Description

Panel for **SLI, top 20% vs bottom 50%** in **Intact Control>Kir** flies across the two **HTL chamber variants**: **flat** and **agarose**.

For this analysis, the reward preference index (RPI) for a fly is defined as:

$$\mathrm{RPI} = \frac{N_{\mathrm{rwd}} - N_{\mathrm{ctrl}}}{N_{\mathrm{rwd}} + N_{\mathrm{ctrl}}}$$

The spatial learning index (SLI) is then defined as:

$$\mathrm{SLI} = \mathrm{RPI}_{\mathrm{exp}} - \mathrm{RPI}_{\mathrm{yok}}$$

The baseline PI used here is:

$$\mathrm{Baseline\ PI} = \mathrm{RPI}^{\mathrm{pre}}_{\mathrm{exp}} - \mathrm{RPI}^{\mathrm{pre}}_{\mathrm{yok}}$$

This panel uses the same learner split definition as the other top-versus-bottom SLI panels, defining the **top 20%** and **bottom 50%** from the mean SLI in **training 2**, averaged across sync buckets **2-5** after skipping the first sync bucket.

The intended interpretation is that the **top 20%** learners perform similarly in both HTL chamber variants, whereas the **bottom 50%** group is near zero in the **flat** chamber but shows substantially higher SLI in the **agarose** chamber, consistent with the agarose task being easier.

This panel includes two plotting variants:

- flat HTL chamber
- agarose HTL chamber

These are one-shot `analyze.py` commands that generate the figures directly. Because the script writes the default filename `reward_pi_diff__10_min_buckets_bottom50_top20.png`, the notebook uses the standard rename helper to preserve both outputs.


In [ ]:
panel_18 = {
    "title": "Panel 18 - Intact Control>Kir flies, HTL chamber SLI, top 20% vs bottom 50%",
    "description": (
        "Compares top-20% and bottom-50% learner SLI traces for Intact "
        "Control>Kir flies between the flat and agarose HTL chamber "
        "variants, using the training-2 mean SLI across sync buckets 2-5 "
        "after skipping the first sync bucket to define learner groups."
    ),
    "export_commands": [],
    "plot_commands": [
        {
            "label": "flat HTL chamber, top 20% vs bottom 50%",
            "command": "python analyze.py -v \"/media/Synology4/Yang Chen/2024-03-04/c3[12]_*,/media/Synology4/Yang Chen/2024-03-04/c4[12]_*,/media/Synology4/Yang Chen/2024-03-14/c4[12]_*,/media/Synology4/Yang Chen/2024-03-18/c5[12]_*,/media/Synology4/Yang Chen/2024-03-18/c6[12]_*,/media/Synology4/Yang Chen/2024-06-12/c[12]_*\" -f 0-9 --rmCC 5 --top-sli-fraction 0.2 --bottom-sli-fraction 0.5 --sli-select-skip-first-sync-buckets 1 --best-worst-trn 2 --best-worst-extreme both --sli-use-training-mean",
            "plot_rewards_xlabel": None,
            "plot_rewards_ylabel": None,
            "rename_outputs": [
                {
                    "from": "imgs/reward_pi_diff__10_min_buckets_bottom50_top20.png",
                    "to": "imgs/rpi_diff__10_min_buckets_bottom50_top20_intact_ctrlKir_flatHTL_2026-04-14.png"
                }
            ],
            "output_path": "imgs/rpi_diff__10_min_buckets_bottom50_top20_intact_ctrlKir_flatHTL_2026-04-14.png"
        },
        {
            "label": "agarose HTL chamber, top 20% vs bottom 50%",
            "command": "python analyze.py -v \"/media/Synology4/Yang Chen/2024-03-1[34]/c5[12]_*,/media/Synology4/Yang Chen/2024-03-1[347]/c6[12]_*,/media/Synology4/Yang Chen/2024-03-18/c[12]_*\" -f 0-9 --rmCC 5 --top-sli-fraction 0.2 --bottom-sli-fraction 0.5 --sli-select-skip-first-sync-buckets 1 --best-worst-trn 2 --best-worst-extreme both --sli-use-training-mean",
            "plot_rewards_xlabel": None,
            "plot_rewards_ylabel": None,
            "rename_outputs": [
                {
                    "from": "imgs/reward_pi_diff__10_min_buckets_bottom50_top20.png",
                    "to": "imgs/rpi_diff__10_min_buckets_bottom50_top20_intact_ctrlKir_agaroseHTL_2026-04-14.png"
                }
            ],
            "output_path": "imgs/rpi_diff__10_min_buckets_bottom50_top20_intact_ctrlKir_agaroseHTL_2026-04-14.png"
        }
    ],
}

panel_18

### Panel 18 Data Export Commands


In [ ]:
RUN_PANEL_18_EXPORTS = False
run_stage("Panel 18: export bundles", panel_18["export_commands"], run=RUN_PANEL_18_EXPORTS)


### Panel 18 Plotting Commands


In [ ]:
RUN_PANEL_18_PLOTS = False
run_stage("Panel 18: generate plot", panel_18["plot_commands"], run=RUN_PANEL_18_PLOTS)


## Panel 19

### Panel 19 Plot Description

Panel for **correlations between SLI and related metrics** in **Intact Control>Kir** flies across the two **HTL chamber variants**: **flat** and **agarose**.

This panel uses the same spatial learning index (SLI) definition established for the other HTL SLI panels in this notebook. It compares three relationships in each chamber variant:

- **baseline SLI** (pre-training SLI) versus later SLI, as a rough estimate of whether pre-training bias predicts later task performance
- **early SLI** versus later SLI, as a readout of how stable learning performance remains across training
- **reward rate** versus SLI, as a test of whether obtaining rewards quickly serves as a proxy for spatial learning

The intended interpretation is that **baseline SLI versus later SLI** shows a stronger correlation in the **agarose** chamber, whereas both **early SLI versus later SLI** and **reward rate versus SLI** show stronger linear relationships in the **flat** chamber. A plausible qualitative summary is that the **agarose** chamber allows a broader range of successful strategies, which pulls weaker performers into a more central cluster, while the **flat** chamber penalizes poor spatial strategies more consistently.

This panel includes two plotting variants:

- flat HTL chamber correlations
- agarose HTL chamber correlations

These are one-shot `analyze.py` commands, and they are the same analysis commands used for **Panel 17**. Each run writes three figure files with default names, so the notebook uses the standard rename helper to preserve and display all outputs for each chamber variant.


In [ ]:
panel_19 = {
    "title": "Panel 19 - Intact Control>Kir flies, HTL chamber SLI, correlations",
    "description": (
        "Shows three SLI-related correlation analyses for Intact "
        "Control>Kir flies in the flat and agarose HTL chamber variants: "
        "baseline SLI versus later SLI, early SLI versus later SLI, and "
        "reward rate versus SLI."
    ),
    "export_commands": [],
    "plot_commands": [
        {
            "label": "flat HTL chamber correlations",
            "command": "python analyze.py -v \"/media/Synology4/Yang Chen/2024-03-04/c3[12]_*,/media/Synology4/Yang Chen/2024-03-04/c4[12]_*,/media/Synology4/Yang Chen/2024-03-14/c4[12]_*,/media/Synology4/Yang Chen/2024-03-18/c5[12]_*,/media/Synology4/Yang Chen/2024-03-18/c6[12]_*,/media/Synology4/Yang Chen/2024-06-12/c[12]_*\" -f 0-9 --rmCC 5 --top-sli-fraction 0.2 --bottom-sli-fraction 0.5 --sli-select-skip-first-sync-buckets 1 --best-worst-trn 2 --best-worst-extreme both --sli-use-training-mean",
            "corr_pre_reward_pi_vs_sli_xlabel": None,
            "corr_pre_reward_pi_vs_sli_ylabel": None,
            "corr_fast_vs_strong_xlabel": None,
            "corr_fast_vs_strong_ylabel": None,
            "corr_sli_vs_rpt_xlabel": None,
            "corr_sli_vs_rpt_ylabel": None,
            "rename_outputs": [
                {
                    "from": "imgs/correlations/corr_pre_reward_pi_vs_sli.png",
                    "to": "imgs/corr_pre_reward_pi_vs_sli_intact_ctrlKir_flatHTL_2026-04-14.png"
                },
                {
                    "from": "imgs/correlations/scatter_fast_vs_strong.png",
                    "to": "imgs/scatter_fast_vs_strong_intact_ctrlKir_flatHTL_2026-04-14.png"
                },
                {
                    "from": "imgs/correlations/corr_sli_vs_rpt_sliT2_mean_skip1__rptT2_mean_skip1.png",
                    "to": "imgs/corr_sli_vs_rpt_T2_intact_ctrlKir_flatHTL_2026-04-14.png"
                }
            ],
            "output_paths": [
                "imgs/corr_pre_reward_pi_vs_sli_intact_ctrlKir_flatHTL_2026-04-14.png",
                "imgs/scatter_fast_vs_strong_intact_ctrlKir_flatHTL_2026-04-14.png",
                "imgs/corr_sli_vs_rpt_T2_intact_ctrlKir_flatHTL_2026-04-14.png"
            ]
        },
        {
            "label": "agarose HTL chamber correlations",
            "command": "python analyze.py -v \"/media/Synology4/Yang Chen/2024-03-1[34]/c5[12]_*,/media/Synology4/Yang Chen/2024-03-1[347]/c6[12]_*,/media/Synology4/Yang Chen/2024-03-18/c[12]_*\" -f 0-9 --rmCC 5 --top-sli-fraction 0.2 --bottom-sli-fraction 0.5 --sli-select-skip-first-sync-buckets 1 --best-worst-trn 2 --best-worst-extreme both --sli-use-training-mean",
            "corr_pre_reward_pi_vs_sli_xlabel": None,
            "corr_pre_reward_pi_vs_sli_ylabel": None,
            "corr_fast_vs_strong_xlabel": None,
            "corr_fast_vs_strong_ylabel": None,
            "corr_sli_vs_rpt_xlabel": None,
            "corr_sli_vs_rpt_ylabel": None,
            "rename_outputs": [
                {
                    "from": "imgs/correlations/corr_pre_reward_pi_vs_sli.png",
                    "to": "imgs/corr_pre_reward_pi_vs_sli_intact_ctrlKir_agaroseHTL_2026-04-14.png"
                },
                {
                    "from": "imgs/correlations/scatter_fast_vs_strong.png",
                    "to": "imgs/scatter_fast_vs_strong_intact_ctrlKir_agaroseHTL_2026-04-14.png"
                },
                {
                    "from": "imgs/correlations/corr_sli_vs_rpt_sliT2_mean_skip1__rptT2_mean_skip1.png",
                    "to": "imgs/corr_sli_vs_rpt_T2_intact_ctrlKir_agaroseHTL_2026-04-14.png"
                }
            ],
            "output_paths": [
                "imgs/corr_pre_reward_pi_vs_sli_intact_ctrlKir_agaroseHTL_2026-04-14.png",
                "imgs/scatter_fast_vs_strong_intact_ctrlKir_agaroseHTL_2026-04-14.png",
                "imgs/corr_sli_vs_rpt_T2_intact_ctrlKir_agaroseHTL_2026-04-14.png"
            ]
        }
    ],
}

panel_19

### Panel 19 Data Export Commands


In [ ]:
RUN_PANEL_19_EXPORTS = False
run_stage("Panel 19: export bundles", panel_19["export_commands"], run=RUN_PANEL_19_EXPORTS)


### Panel 19 Plotting Commands


In [ ]:
RUN_PANEL_19_PLOTS = False
run_stage("Panel 19: generate plot", panel_19["plot_commands"], run=RUN_PANEL_19_PLOTS)


## Panel 20

### Panel 20 Plot Description

Panel for **correlations between SLI and related metrics** in **Intact Control>Kir** flies across the two **large-chamber variants**: **flat** and **agarose**.

This panel uses the same spatial learning index (SLI) definition established for the other large-chamber SLI panels in this notebook. It compares three relationships in each chamber variant:

- **pre-training exploration** (fraction of the floor explored in pre-training) versus later SLI, as a rough estimate of whether an existing spatial sense of the chamber layout aligns with later task performance
- **early SLI** versus later SLI, as a readout of how stable learning performance remains across training
- **reward rate** versus SLI, as a test of whether obtaining rewards quickly serves as a proxy for spatial learning

The intended interpretation is that **pre-training exploration versus later SLI** shows a strong correlation in the **flat** large chamber but not in the **agarose** large chamber, consistent with the idea that mapping the chamber floor matters more when fewer additional landmarks are available. Both **early SLI versus later SLI** and **reward rate versus SLI** also show stronger linear relationships in the **flat** chamber. A plausible qualitative summary is that the **agarose** chamber supports additional successful strategies, which weakens the dependence of later SLI on early spatial mapping and allows reward rate to extend upward even after SLI has begun to saturate.

This panel includes two plotting variants:

- flat large chamber correlations
- agarose large chamber correlations

These are one-shot `analyze.py` commands with no separate export stage. Each run writes three figure files with default names, so the notebook uses the standard rename helper to preserve and display all outputs for each chamber variant.


In [ ]:
panel_20 = {
    "title": "Panel 20 - Intact Control>Kir flies, large chamber SLI, correlations",
    "description": (
        "Shows three SLI-related correlation analyses for Intact "
        "Control>Kir flies in the flat and agarose large-chamber variants: "
        "pre-training exploration versus later SLI, early SLI versus later "
        "SLI, and reward rate versus SLI."
    ),
    "export_commands": [],
    "plot_commands": [
        {
            "label": "flat large chamber correlations",
            "command": "python analyze.py -v \"/media/Synology4/Yang Chen/2025-05-30/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-05-30/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-06-02/c5[12]_*,/media/Synology4/Yang Chen/2025-06-29/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-06-30/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-06-30/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-0[367]/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-07-0[367]/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-0[45]/c5[12]_*,/media/Synology4/Yang Chen/2025-07-05/c6[12]_*,/media/Synology4/Yang Chen/2025-07-11/c5[12]_*,/media/Synology4/Yang Chen/2025-07-11/c6[12]_*,/media/Synology4/Yang Chen/2025-07-26/afternoon/c3[12]_*,/media/Synology4/Yang Chen/2025-07-26/afternoon/c6[12]_*\" -f 0-1 --rCC 15 --top-sli-fraction 0.2 --bottom-sli-fraction 0.5 --sli-select-skip-first-sync-buckets 1 --best-worst-trn 2 --best-worst-extreme both --sli-use-training-mean",
            "corr_pre_floor_exploration_vs_sli_xlabel": None,
            "corr_pre_floor_exploration_vs_sli_ylabel": None,
            "corr_fast_vs_strong_xlabel": None,
            "corr_fast_vs_strong_ylabel": None,
            "corr_sli_vs_rpt_xlabel": None,
            "corr_sli_vs_rpt_ylabel": None,
            "rename_outputs": [
                {
                    "from": "imgs/correlations/corr_pre_floor_exploration_vs_sli_final.png",
                    "to": "imgs/corr_pre_floor_exploration_vs_sli_final_intact_ctrlKir_flatLgc_2026-04-14.png"
                },
                {
                    "from": "imgs/correlations/scatter_fast_vs_strong.png",
                    "to": "imgs/scatter_fast_vs_strong_intact_ctrlKir_flatLgc_2026-04-14.png"
                },
                {
                    "from": "imgs/correlations/corr_sli_vs_rpt_sliT2_mean_skip1__rptT2_mean_skip1.png",
                    "to": "imgs/corr_sli_vs_rpt_T2_intact_ctrlKir_flatLgc_2026-04-14.png"
                }
            ],
            "output_paths": [
                "imgs/corr_pre_floor_exploration_vs_sli_final_intact_ctrlKir_flatLgc_2026-04-14.png",
                "imgs/scatter_fast_vs_strong_intact_ctrlKir_flatLgc_2026-04-14.png",
                "imgs/corr_sli_vs_rpt_T2_intact_ctrlKir_flatLgc_2026-04-14.png"
            ]
        },
        {
            "label": "agarose large chamber correlations",
            "command": "python analyze.py -v \"/media/Synology4/Yang Chen/2025-06-28/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-10/night/c5[12]_*,/media/Synology4/Yang Chen/2025-07-10/night/c6[12]_*,/media/Synology4/Yang Chen/2025-07-1[26]/c6[12]_*,/media/Synology4/Yang Chen/2025-07-1[26]/c3[12]_*,/media/Synology4/Yang Chen/2025-07-1[36]/afternoon/c3[12]_*,/media/Synology4/Yang Chen/2025-07-1[36]/afternoon/c4[12]_*,/media/Synology4/Yang Chen/2025-07-1[6]/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-07-1[6]/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-16/c4[12]_*,/media/Synology4/Yang Chen/2025-07-16/c5[12]_*,/media/Synology4/Yang Chen/2025-07-2[25]/night/c5[12]_*,/media/Synology4/Yang Chen/2025-07-2[5]/night/c3[12]_*,/media/Synology4/Yang Chen/2025-07-2[5]/night/c4[12]_*,/media/Synology4/Yang Chen/2025-07-2[5]/night/c6[12]_*\" -f 0-1 --rCC 15 --top-sli-fraction 0.2 --bottom-sli-fraction 0.5 --sli-select-skip-first-sync-buckets 1 --best-worst-trn 2 --best-worst-extreme both --sli-use-training-mean",
            "corr_pre_floor_exploration_vs_sli_xlabel": None,
            "corr_pre_floor_exploration_vs_sli_ylabel": None,
            "corr_fast_vs_strong_xlabel": None,
            "corr_fast_vs_strong_ylabel": None,
            "corr_sli_vs_rpt_xlabel": None,
            "corr_sli_vs_rpt_ylabel": None,
            "rename_outputs": [
                {
                    "from": "imgs/correlations/corr_pre_floor_exploration_vs_sli_final.png",
                    "to": "imgs/corr_pre_floor_exploration_vs_sli_final_intact_ctrlKir_agaroseLgc_2026-04-14.png"
                },
                {
                    "from": "imgs/correlations/scatter_fast_vs_strong.png",
                    "to": "imgs/scatter_fast_vs_strong_intact_ctrlKir_agaroseLgc_2026-04-14.png"
                },
                {
                    "from": "imgs/correlations/corr_sli_vs_rpt_sliT2_mean_skip1__rptT2_mean_skip1.png",
                    "to": "imgs/corr_sli_vs_rpt_T2_intact_ctrlKir_agaroseLgc_2026-04-14.png"
                }
            ],
            "output_paths": [
                "imgs/corr_pre_floor_exploration_vs_sli_final_intact_ctrlKir_agaroseLgc_2026-04-14.png",
                "imgs/scatter_fast_vs_strong_intact_ctrlKir_agaroseLgc_2026-04-14.png",
                "imgs/corr_sli_vs_rpt_T2_intact_ctrlKir_agaroseLgc_2026-04-14.png"
            ]
        }
    ],
}

panel_20

### Panel 20 Data Export Commands


In [ ]:
RUN_PANEL_20_EXPORTS = False
run_stage("Panel 20: export bundles", panel_20["export_commands"], run=RUN_PANEL_20_EXPORTS)


### Panel 20 Plotting Commands


In [ ]:
RUN_PANEL_20_PLOTS = False
run_stage("Panel 20: generate plot", panel_20["plot_commands"], run=RUN_PANEL_20_PLOTS)


## Panel 21

### Panel 21 Plot Description

Panel for **wall-contact events per between-reward interval** in the **flat large chamber**, comparing **Intact Control>Kir**, **Intact PFN>Kir**, and **AR Control>Kir** flies across **trainings 1 and 2**.

The plotted quantity is the **mean number of wall-contact events per between-reward interval**, computed per fly and then averaged across flies within each group, with **95% confidence interval** whiskers. Sync bucket **1** is excluded from both the exported metric and the learner-selection definition.

This panel includes two plotting variants:

- all learners
- top 20% learners, where the learner split is defined from the **mean SLI in training 2** after skipping the first sync bucket

The intended interpretation is that in **training 1**, where the reward circle is larger and the task is easier, **Intact Control>Kir** flies show the fewest wall-contact events per between-reward interval, followed by **AR Control>Kir** flies, while **Intact PFN>Kir** flies show the most. In **training 2**, when the reward circle is smaller, the ordering flips: **AR Control>Kir** flies show significantly more wall-contact events per interval than either of the other two groups.


In [ ]:
panel_21 = {
    "title": "Panel 21 - Wall contacts per between-reward interval, flat large chamber",
    "description": (
        "Compares mean wall-contact event counts per between-reward interval "
        "between Intact Control>Kir, Intact PFN>Kir, "
        "and AR Control>Kir flies across trainings 1 and 2 in the "
        "flat large chamber, for both all learners and the top 20% of "
        "learners defined from training-2 mean SLI."
    ),
    "export_commands": [
        {
            "label": "Intact Control>Kir, all learners",
            "command": "python analyze.py -v '/media/Synology4/Yang Chen/2025-05-30/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-05-30/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-06-02/c5[12]_*,/media/Synology4/Yang Chen/2025-06-29/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-06-30/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-06-30/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-0[367]/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-07-0[367]/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-0[45]/c5[12]_*,/media/Synology4/Yang Chen/2025-07-05/c6[12]_*,/media/Synology4/Yang Chen/2025-07-11/c5[12]_*,/media/Synology4/Yang Chen/2025-07-11/c6[12]_*,/media/Synology4/Yang Chen/2025-07-26/afternoon/c3[12]_*,/media/Synology4/Yang Chen/2025-07-26/afternoon/c6[12]_*' -f 0-1 --rCC 15 --wall-contacts-per-reward-interval-total-export exports/wall_contact_per_rwd_interval_intact_ctrlKir_flat_lgc_T2_2026-04-20.npz --wall-contacts-per-reward-interval-total-trainings 2 --wall-contacts-per-reward-interval-total-ci --skip-first-sync-buckets 1 --sli-select-skip-first-sync-buckets 1 --sli-use-training-mean --best-worst-trn 2"
        },
        {
            "label": "Intact Control>Kir, top 20% learners",
            "command": "python analyze.py -v '/media/Synology4/Yang Chen/2025-05-30/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-05-30/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-06-02/c5[12]_*,/media/Synology4/Yang Chen/2025-06-29/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-06-30/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-06-30/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-0[367]/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-07-0[367]/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-0[45]/c5[12]_*,/media/Synology4/Yang Chen/2025-07-05/c6[12]_*,/media/Synology4/Yang Chen/2025-07-11/c5[12]_*,/media/Synology4/Yang Chen/2025-07-11/c6[12]_*,/media/Synology4/Yang Chen/2025-07-26/afternoon/c3[12]_*,/media/Synology4/Yang Chen/2025-07-26/afternoon/c6[12]_*' -f 0-1 --rCC 15 --wall-contacts-per-reward-interval-total-export exports/wall_contact_per_rwd_interval_intact_ctrlKir_flat_lgc_T2_top_2026-04-20.npz --wall-contacts-per-reward-interval-total-trainings 2 --wall-contacts-per-reward-interval-total-ci --skip-first-sync-buckets 1 --sli-select-skip-first-sync-buckets 1 --sli-use-training-mean --best-worst-trn 2 --wall-contacts-per-reward-interval-total-sli-group top --top-sli-fraction 0.2"
        },
        {
            "label": "AR Control>Kir, all learners",
            "command": "python analyze.py -v '/media/Synology4/Yang Chen/2025-07-17/c4[12]_*,/media/Synology4/Yang Chen/2025-07-17/c5[12]_*,/media/Synology4/Yang Chen/2025-07-17/c6[12]_*,/media/Synology4/Yang Chen/2025-07-18/night/c5[12]_*,/media/Synology4/Yang Chen/2025-07-18/night/c6[12]_*,/media/Synology4/Yang Chen/2025-07-2[09]/night/c3[12]_*,/media/Synology4/Yang Chen/2025-07-2[01]/night/c4[12]_*,/media/Synology4/Yang Chen/2025-07-21/night/c6[12]_*,/media/Synology4/Yang Chen/2025-07-27/c6[12]_*,/media/Synology4/Yang Chen/2025-07-27/c5[12]_*,/media/Synology4/Yang Chen/2025-07-27/afternoon/c3[12]_*,/media/Synology4/Yang Chen/2025-07-27/afternoon/c4[12]_*,/media/Synology4/Yang Chen/2025-07-2[79]/night/c4[12]_*,/media/Synology4/Yang Chen/2025-07-30/night/c3[12]_*,/media/Synology4/Yang Chen/2025-07-30/night/c4[12]_*,/media/Synology4/Yang Chen/2025-08-02/night/c3[12]_*' -f 0-1 --rCC 15 --wall-contacts-per-reward-interval-total-export exports/wall_contact_per_rwd_interval_ar_ctrlKir_flat_lgc_T2_2026-04-20.npz --wall-contacts-per-reward-interval-total-trainings 2 --wall-contacts-per-reward-interval-total-ci --skip-first-sync-buckets 1 --sli-select-skip-first-sync-buckets 1 --sli-use-training-mean --best-worst-trn 2"
        },
        {
            "label": "AR Control>Kir, top 20% learners",
            "command": "python analyze.py -v '/media/Synology4/Yang Chen/2025-07-17/c4[12]_*,/media/Synology4/Yang Chen/2025-07-17/c5[12]_*,/media/Synology4/Yang Chen/2025-07-17/c6[12]_*,/media/Synology4/Yang Chen/2025-07-18/night/c5[12]_*,/media/Synology4/Yang Chen/2025-07-18/night/c6[12]_*,/media/Synology4/Yang Chen/2025-07-2[09]/night/c3[12]_*,/media/Synology4/Yang Chen/2025-07-2[01]/night/c4[12]_*,/media/Synology4/Yang Chen/2025-07-21/night/c6[12]_*,/media/Synology4/Yang Chen/2025-07-27/c6[12]_*,/media/Synology4/Yang Chen/2025-07-27/c5[12]_*,/media/Synology4/Yang Chen/2025-07-27/afternoon/c3[12]_*,/media/Synology4/Yang Chen/2025-07-27/afternoon/c4[12]_*,/media/Synology4/Yang Chen/2025-07-2[79]/night/c4[12]_*,/media/Synology4/Yang Chen/2025-07-30/night/c3[12]_*,/media/Synology4/Yang Chen/2025-07-30/night/c4[12]_*,/media/Synology4/Yang Chen/2025-08-02/night/c3[12]_*' -f 0-1 --rCC 15 --wall-contacts-per-reward-interval-total-export exports/wall_contact_per_rwd_interval_ar_ctrlKir_flat_lgc_T2_top_2026-04-20.npz --wall-contacts-per-reward-interval-total-trainings 2 --wall-contacts-per-reward-interval-total-ci --skip-first-sync-buckets 1 --sli-select-skip-first-sync-buckets 1 --sli-use-training-mean --best-worst-trn 2 --wall-contacts-per-reward-interval-total-sli-group top --top-sli-fraction 0.2"
        },
        {
            "label": "Intact PFN>Kir, all learners",
            "command": "python analyze.py -v '/media/Synology4/Yang Chen/2025-05-31/c5[12]_*,/media/Synology4/Yang Chen/2025-05-31/c6[12]_*,/media/Synology4/Yang Chen/2025-05-31/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-06-05/c6[12]_*,/media/Synology4/Yang Chen/2025-06-28/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-06-29/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-0[124]/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-07-0[124]/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-06/c6[12]_*,/media/Synology4/Yang Chen/2025-07-12/night/c32_*,/media/Synology4/Yang Chen/2025-07-21/night/c5[12]_*,/media/Synology4/Yang Chen/2025-07-22/night/c3[12]_*' -f 0-1 --rCC 15 --wall-contacts-per-reward-interval-total-export exports/wall_contact_per_rwd_interval_intact_pfnKir_flat_lgc_T2_2026-04-20.npz --wall-contacts-per-reward-interval-total-trainings 2 --wall-contacts-per-reward-interval-total-ci --skip-first-sync-buckets 1 --sli-select-skip-first-sync-buckets 1 --sli-use-training-mean --best-worst-trn 2"
        },
        {
            "label": "Intact PFN>Kir, top 20% learners",
            "command": "python analyze.py -v '/media/Synology4/Yang Chen/2025-05-31/c5[12]_*,/media/Synology4/Yang Chen/2025-05-31/c6[12]_*,/media/Synology4/Yang Chen/2025-05-31/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-06-05/c6[12]_*,/media/Synology4/Yang Chen/2025-06-28/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-06-29/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-0[124]/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-07-0[124]/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-06/c6[12]_*,/media/Synology4/Yang Chen/2025-07-12/night/c32_*,/media/Synology4/Yang Chen/2025-07-21/night/c5[12]_*,/media/Synology4/Yang Chen/2025-07-22/night/c3[12]_*' -f 0-1 --rCC 15 --wall-contacts-per-reward-interval-total-export exports/wall_contact_per_rwd_interval_intact_pfnKir_flat_lgc_T2_top_2026-04-20.npz --wall-contacts-per-reward-interval-total-trainings 2 --wall-contacts-per-reward-interval-total-ci --skip-first-sync-buckets 1 --sli-select-skip-first-sync-buckets 1 --sli-use-training-mean --best-worst-trn 2 --wall-contacts-per-reward-interval-total-sli-group top --top-sli-fraction 0.2"
        },
    ],
    "plot_commands": [
        {
            "label": "all learners overlay",
            "command": "python analyze.py -v \"/media/Synology4/Yang Chen/2025-05-31/c5[1]_*\" -f 0-1 --rCC 15 --wall-contacts-per-reward-interval-total-import-npz \"Intact Control>Kir:exports/wall_contact_per_rwd_interval_intact_ctrlKir_flat_lgc_T2_2026-04-20.npz\" --wall-contacts-per-reward-interval-total-import-npz \"Intact PFN>Kir:exports/wall_contact_per_rwd_interval_intact_pfnKir_flat_lgc_T2_2026-04-20.npz\" --wall-contacts-per-reward-interval-total-import-npz \"AR Control>Kir:exports/wall_contact_per_rwd_interval_ar_ctrlKir_flat_lgc_T2_2026-04-20.npz\" --wall-contacts-per-reward-interval-total-stats --wall-contacts-per-reward-interval-total-import-out \"exports/wall_contact_per_rwd_interval_intact_ctrlKir_vs_ar_ctrlKir_vs_intact_pfnKir_flat_lgc_T2_2026-04-20.png\"",
            "output_path": "exports/wall_contact_per_rwd_interval_intact_ctrlKir_vs_ar_ctrlKir_vs_intact_pfnKir_flat_lgc_T2_2026-04-20.png"
        },
        {
            "label": "top 20% learners overlay",
            "command": "python analyze.py -v \"/media/Synology4/Yang Chen/2025-05-31/c5[1]_*\" -f 0-1 --rCC 15 --wall-contacts-per-reward-interval-total-import-npz \"Intact Control>Kir:exports/wall_contact_per_rwd_interval_intact_ctrlKir_flat_lgc_T2_top_2026-04-20.npz\" --wall-contacts-per-reward-interval-total-import-npz \"Intact PFN>Kir:exports/wall_contact_per_rwd_interval_intact_pfnKir_flat_lgc_T2_top_2026-04-20.npz\" --wall-contacts-per-reward-interval-total-import-npz \"AR Control>Kir:exports/wall_contact_per_rwd_interval_ar_ctrlKir_flat_lgc_T2_top_2026-04-20.npz\" --wall-contacts-per-reward-interval-total-stats --wall-contacts-per-reward-interval-total-import-out \"exports/wall_contact_per_rwd_interval_intact_ctrlKir_vs_ar_ctrlKir_vs_intact_pfnKir_flat_lgc_T2_top_2026-04-20.png\"",
            "output_path": "exports/wall_contact_per_rwd_interval_intact_ctrlKir_vs_ar_ctrlKir_vs_intact_pfnKir_flat_lgc_T2_top_2026-04-20.png"
        },
    ],
}

panel_21

### Panel 21 Data Export Commands


In [ ]:
RUN_PANEL_21_EXPORTS = False
run_stage("Panel 21: export bundles", panel_21["export_commands"], run=RUN_PANEL_21_EXPORTS)


### Panel 21 Plotting Commands


In [ ]:
RUN_PANEL_21_PLOTS = False
run_stage("Panel 21: generate plot", panel_21["plot_commands"], run=RUN_PANEL_21_PLOTS)


## Panel 22

### Panel 22 Plot Description

Panel for **binned turnback ratio** in the **flat large chamber**, comparing the same three cohorts used in **Panel 1**: **Intact Control>Kir**, **Intact PFN>Kir**, and **AR Control>Kir**.

This panel uses the **turnback-excursion bin** analysis, which starts each episode when a fly exits an inner circle around the reward and classifies the episode as a successful turnback if the fly re-enters that inner circle before leaving the outer radius associated with the current bin. Right-censored episodes are excluded from the denominator.

Here, the inner-circle radial delta is **4 mm**, and the outer-radius bins are **4-8 mm**, **8-16 mm**, and **16-24 mm**, all measured relative to the reward circle. The analysis is restricted to **training 2**, and the learner-selection settings match the other large-chamber comparison panels by using the **training-2 mean SLI** after skipping the first sync bucket.

The intended interpretation is that turnback success increases across larger excursion bins for all three groups, but **Intact Control>Kir** flies stay highest in each bin, **Intact PFN>Kir** flies are intermediate, and **AR Control>Kir** flies are lowest.


In [ ]:
panel_22 = {
    "title": "Panel 22 - Binned turnback ratio",
    "description": (
        "Compares training-2 binned turnback ratio in the flat large "
        "chamber between Intact Control>Kir, Intact "
        "PFN>Kir, and AR Control>Kir flies, using a 4 mm "
        "inner-circle delta and outer-radius bins of 4-8 mm, 8-16 mm, "
        "and 16-24 mm."
    ),
    "export_commands": [
        {
            "label": "Intact Control>Kir",
            "command": "python analyze.py -v '/media/Synology4/Yang Chen/2025-05-30/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-05-30/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-06-02/c5[12]_*,/media/Synology4/Yang Chen/2025-06-29/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-06-30/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-06-30/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-0[367]/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-07-0[367]/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-0[45]/c5[12]_*,/media/Synology4/Yang Chen/2025-07-05/c6[12]_*,/media/Synology4/Yang Chen/2025-07-11/c5[12]_*,/media/Synology4/Yang Chen/2025-07-11/c6[12]_*,/media/Synology4/Yang Chen/2025-07-26/afternoon/c3[12]_*,/media/Synology4/Yang Chen/2025-07-26/afternoon/c6[12]_*' -f 0-1 --rCC 15 --export-turnback-excursion-bin-sli-bundle exports/turnback_binned_intact_ctrlKir_flatLgc_T2_2026-04-20.npz --turnback-inner-delta-mm 4 --turnback-excursion-bin-trainings 2 --turnback-excursion-bin-skip-first-sync-buckets 1 --turnback-excursion-bin-edges-mm 4,8,16,24 --best-worst-trn 2 --sli-use-training-mean --sli-select-skip-first-sync-buckets 1 --export-group-label \"Intact Control>Kir\""
        },
        {
            "label": "Intact PFN>Kir",
            "command": "python analyze.py -v '/media/Synology4/Yang Chen/2025-05-31/c5[12]_*,/media/Synology4/Yang Chen/2025-05-31/c6[12]_*,/media/Synology4/Yang Chen/2025-05-31/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-06-05/c6[12]_*,/media/Synology4/Yang Chen/2025-06-28/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-06-29/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-0[124]/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-07-0[124]/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-06/c6[12]_*,/media/Synology4/Yang Chen/2025-07-12/night/c32_*,/media/Synology4/Yang Chen/2025-07-21/night/c5[12]_*,/media/Synology4/Yang Chen/2025-07-22/night/c3[12]_*' -f 0-1 --rCC 15 --export-turnback-excursion-bin-sli-bundle exports/turnback_binned_intact_pfnKir_flatLgc_T2_2026-04-20.npz --turnback-inner-delta-mm 4 --turnback-excursion-bin-trainings 2 --turnback-excursion-bin-skip-first-sync-buckets 1 --turnback-excursion-bin-edges-mm 4,8,16,24 --best-worst-trn 2 --sli-use-training-mean --sli-select-skip-first-sync-buckets 1 --export-group-label \"Intact PFN>Kir\""
        },
        {
            "label": "AR Control>Kir",
            "command": "python analyze.py -v '/media/Synology4/Yang Chen/2025-07-17/c4[12]_*,/media/Synology4/Yang Chen/2025-07-17/c5[12]_*,/media/Synology4/Yang Chen/2025-07-17/c6[12]_*,/media/Synology4/Yang Chen/2025-07-18/night/c5[12]_*,/media/Synology4/Yang Chen/2025-07-18/night/c6[12]_*,/media/Synology4/Yang Chen/2025-07-2[09]/night/c3[12]_*,/media/Synology4/Yang Chen/2025-07-2[01]/night/c4[12]_*,/media/Synology4/Yang Chen/2025-07-21/night/c6[12]_*,/media/Synology4/Yang Chen/2025-07-27/c6[12]_*,/media/Synology4/Yang Chen/2025-07-27/c5[12]_*,/media/Synology4/Yang Chen/2025-07-27/afternoon/c3[12]_*,/media/Synology4/Yang Chen/2025-07-27/afternoon/c4[12]_*,/media/Synology4/Yang Chen/2025-07-2[79]/night/c4[12]_*,/media/Synology4/Yang Chen/2025-07-30/night/c3[12]_*,/media/Synology4/Yang Chen/2025-07-30/night/c4[12]_*,/media/Synology4/Yang Chen/2025-08-02/night/c3[12]_*' -f 0-1 --rCC 15 --export-turnback-excursion-bin-sli-bundle exports/turnback_binned_ar_ctrlKir_flatLgc_T2_2026-04-20.npz --turnback-inner-delta-mm 4 --turnback-excursion-bin-trainings 2 --turnback-excursion-bin-skip-first-sync-buckets 1 --turnback-excursion-bin-edges-mm 4,8,16,24 --best-worst-trn 2 --sli-use-training-mean --sli-select-skip-first-sync-buckets 1 --export-group-label \"AR Control>Kir\""
        },
    ],
    "plot_commands": [
        {
            "label": "training-2 binned turnback ratio comparison",
            "command": "python -m scripts.plot_turnback_excursion_bin_sli_bundles --bundles exports/turnback_binned_intact_ctrlKir_flatLgc_T2_2026-04-20.npz,exports/turnback_binned_intact_pfnKir_flatLgc_T2_2026-04-20.npz,exports/turnback_binned_ar_ctrlKir_flatLgc_T2_2026-04-20.npz --out exports/turnback_binned_intact_ctrlKir_vs_intact_pfnKir_vs_ar_ctrlKir_flatLgc_T2_2026-04-20.png --stats",
            "xlabel": None,
            "ylabel": None,
            "output_path": "exports/turnback_binned_intact_ctrlKir_vs_intact_pfnKir_vs_ar_ctrlKir_flatLgc_T2_2026-04-20.png"
        },
    ],
}

panel_22


### Panel 22 Data Export Commands


In [ ]:
RUN_PANEL_22_EXPORTS = False
run_stage("Panel 22: export bundles", panel_22["export_commands"], run=RUN_PANEL_22_EXPORTS)


### Panel 22 Plotting Commands


In [ ]:
RUN_PANEL_22_PLOTS = False
run_stage("Panel 22: generate plot", panel_22["plot_commands"], run=RUN_PANEL_22_PLOTS)


## Panel 23

### Panel 23 Plot Description

Panel for **SLI versus first-10 rewards rate** in **Intact Control>Kir** flies across the two **large chamber variants**: **flat** and **agarose**.

For this analysis, the reward preference index (RPI) for a fly is defined as:

$$\mathrm{RPI} = \frac{N_{\mathrm{rwd}} - N_{\mathrm{ctrl}}}{N_{\mathrm{rwd}} + N_{\mathrm{ctrl}}}$$

The spatial learning index (SLI) is then defined as:

$$\mathrm{SLI} = \mathrm{RPI}_{\mathrm{exp}} - \mathrm{RPI}_{\mathrm{yok}}$$

The baseline PI used here is:

$$\mathrm{Baseline\ PI} = \mathrm{RPI}^{\mathrm{pre}}_{\mathrm{exp}} - \mathrm{RPI}^{\mathrm{pre}}_{\mathrm{yok}}$$

This panel shows two related correlation variants between SLI and the reward rate computed from the first **10 calculated rewards** in **training 1**, repeated for both large chamber variants:

- SLI from the first sync bucket of training 1
- SLI averaged across training 1 sync buckets 2-5

Each variant is run for:

- flat large chamber
- agarose large chamber

These are one-shot `analyze.py` commands that generate the scatter plots directly, so there is no separate bundle export stage for this panel.


In [ ]:
panel_23 = {
    "title": "Panel 23 - SLI versus first-10 rewards rate, intact controls, large chambers",
    "description": (
        "Shows two correlation views between spatial learning index (SLI) and "
        "the reward rate for the first 10 calculated rewards in Intact "
        "Control>Kir flies across the flat and agarose large chamber "
        "variants: one using only the first sync bucket of training 1, and "
        "one using the mean SLI across training 1 sync buckets 2-5."
    ),
    "export_commands": [],
    "plot_commands": [
        {
            "label": "flat large chamber, T1 sync bucket 1",
            "command": "python analyze.py -v '/media/Synology4/Yang Chen/2025-05-30/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-05-30/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-06-02/c5[12]_*,/media/Synology4/Yang Chen/2025-06-29/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-06-30/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-06-30/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-0[367]/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-07-0[367]/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-0[45]/c5[12]_*,/media/Synology4/Yang Chen/2025-07-05/c6[12]_*,/media/Synology4/Yang Chen/2025-07-11/c5[12]_*,/media/Synology4/Yang Chen/2025-07-11/c6[12]_*,/media/Synology4/Yang Chen/2025-07-26/afternoon/c3[12]_*,/media/Synology4/Yang Chen/2025-07-26/afternoon/c6[12]_*' -f 0-1 --rCC 15 --first-n-reward-diagnostics --first-n-reward-diagnostics-reward-event-type calc --first-n-reward-diagnostics-pi-threshold 10 --first-n-reward-diagnostics-sli-group all --first-n-reward-diagnostics-trainings 1 --first-n-reward-diagnostics-n 10 --first-n-reward-diagnostics-x-by selected_reward_rate_to_nth_per_min --first-n-reward-diagnostics-y-by sli --first-n-reward-diagnostics-color-by none --best-worst-sli --best-worst-trn 1 --sli-use-training-mean --sli-select-keep-first-sync-buckets 1 --first-n-reward-diagnostics-csv exports/sliT1SB1_vs_rwdsFirst10_intact_ctrlKir_flatLgc_2026-04-22.csv --first-n-reward-diagnostics-plot-out exports/sliT1SB1_vs_rwdsFirst10_intact_ctrlKir_flatLgc_2026-04-22.png",
            "first_n_reward_diagnostics_xlabel": None,
            "first_n_reward_diagnostics_ylabel": None,
            "output_path": "exports/sliT1SB1_vs_rwdsFirst10_intact_ctrlKir_flatLgc_2026-04-22.png"
        },
        {
            "label": "flat large chamber, T1 sync buckets 2-5 mean",
            "command": "python analyze.py -v '/media/Synology4/Yang Chen/2025-05-30/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-05-30/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-06-02/c5[12]_*,/media/Synology4/Yang Chen/2025-06-29/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-06-30/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-06-30/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-0[367]/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-07-0[367]/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-0[45]/c5[12]_*,/media/Synology4/Yang Chen/2025-07-05/c6[12]_*,/media/Synology4/Yang Chen/2025-07-11/c5[12]_*,/media/Synology4/Yang Chen/2025-07-11/c6[12]_*,/media/Synology4/Yang Chen/2025-07-26/afternoon/c3[12]_*,/media/Synology4/Yang Chen/2025-07-26/afternoon/c6[12]_*' -f 0-1 --rCC 15 --first-n-reward-diagnostics --first-n-reward-diagnostics-reward-event-type calc --first-n-reward-diagnostics-pi-threshold 10 --first-n-reward-diagnostics-sli-group all --first-n-reward-diagnostics-trainings 1 --first-n-reward-diagnostics-n 10 --first-n-reward-diagnostics-x-by selected_reward_rate_to_nth_per_min --first-n-reward-diagnostics-y-by sli --first-n-reward-diagnostics-color-by none --best-worst-sli --best-worst-trn 1 --sli-use-training-mean --skip-first-sync-buckets 1 --sli-select-skip-first-sync-buckets 1 --first-n-reward-diagnostics-csv exports/sliT1SB2SB5_vs_rwdsFirst10_intact_ctrlKir_flatLgc_2026-04-22.csv --first-n-reward-diagnostics-plot-out exports/sliT1SB2SB5_vs_rwdsFirst10_intact_ctrlKir_flatLgc_2026-04-22.png",
            "first_n_reward_diagnostics_xlabel": None,
            "first_n_reward_diagnostics_ylabel": None,
            "output_path": "exports/sliT1SB2SB5_vs_rwdsFirst10_intact_ctrlKir_flatLgc_2026-04-22.png"
        },
        {
            "label": "agarose large chamber, T1 sync bucket 1",
            "command": "python analyze.py -v '/media/Synology4/Yang Chen/2025-06-28/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-10/night/c5[12]_*,/media/Synology4/Yang Chen/2025-07-10/night/c6[12]_*,/media/Synology4/Yang Chen/2025-07-1[26]/c6[12]_*,/media/Synology4/Yang Chen/2025-07-1[26]/c3[12]_*,/media/Synology4/Yang Chen/2025-07-1[36]/afternoon/c3[12]_*,/media/Synology4/Yang Chen/2025-07-1[36]/afternoon/c4[12]_*,/media/Synology4/Yang Chen/2025-07-1[6]/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-07-1[6]/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-16/c4[12]_*,/media/Synology4/Yang Chen/2025-07-16/c5[12]_*,/media/Synology4/Yang Chen/2025-07-2[25]/night/c5[12]_*,/media/Synology4/Yang Chen/2025-07-2[5]/night/c3[12]_*,/media/Synology4/Yang Chen/2025-07-2[5]/night/c4[12]_*,/media/Synology4/Yang Chen/2025-07-2[5]/night/c6[12]_*' -f 0-1 --rCC 15 --first-n-reward-diagnostics --first-n-reward-diagnostics-reward-event-type calc --first-n-reward-diagnostics-pi-threshold 10 --first-n-reward-diagnostics-sli-group all --first-n-reward-diagnostics-trainings 1 --first-n-reward-diagnostics-n 10 --first-n-reward-diagnostics-x-by selected_reward_rate_to_nth_per_min --first-n-reward-diagnostics-y-by sli --first-n-reward-diagnostics-color-by none --best-worst-sli --best-worst-trn 1 --sli-use-training-mean --sli-select-keep-first-sync-buckets 1 --first-n-reward-diagnostics-csv exports/sliT1SB1_vs_rwdsFirst10_intact_ctrlKir_agaroseLgc_2026-04-22.csv --first-n-reward-diagnostics-plot-out exports/sliT1SB1_vs_rwdsFirst10_intact_ctrlKir_agaroseLgc_2026-04-22.png",
            "first_n_reward_diagnostics_xlabel": None,
            "first_n_reward_diagnostics_ylabel": None,
            "output_path": "exports/sliT1SB1_vs_rwdsFirst10_intact_ctrlKir_agaroseLgc_2026-04-22.png"
        },
        {
            "label": "agarose large chamber, T1 sync buckets 2-5 mean",
            "command": "python analyze.py -v '/media/Synology4/Yang Chen/2025-06-28/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-10/night/c5[12]_*,/media/Synology4/Yang Chen/2025-07-10/night/c6[12]_*,/media/Synology4/Yang Chen/2025-07-1[26]/c6[12]_*,/media/Synology4/Yang Chen/2025-07-1[26]/c3[12]_*,/media/Synology4/Yang Chen/2025-07-1[36]/afternoon/c3[12]_*,/media/Synology4/Yang Chen/2025-07-1[36]/afternoon/c4[12]_*,/media/Synology4/Yang Chen/2025-07-1[6]/afternoon/c5[12]_*,/media/Synology4/Yang Chen/2025-07-1[6]/afternoon/c6[12]_*,/media/Synology4/Yang Chen/2025-07-16/c4[12]_*,/media/Synology4/Yang Chen/2025-07-16/c5[12]_*,/media/Synology4/Yang Chen/2025-07-2[25]/night/c5[12]_*,/media/Synology4/Yang Chen/2025-07-2[5]/night/c3[12]_*,/media/Synology4/Yang Chen/2025-07-2[5]/night/c4[12]_*,/media/Synology4/Yang Chen/2025-07-2[5]/night/c6[12]_*' -f 0-1 --rCC 15 --first-n-reward-diagnostics --first-n-reward-diagnostics-reward-event-type calc --first-n-reward-diagnostics-pi-threshold 10 --first-n-reward-diagnostics-sli-group all --first-n-reward-diagnostics-trainings 1 --first-n-reward-diagnostics-n 10 --first-n-reward-diagnostics-x-by selected_reward_rate_to_nth_per_min --first-n-reward-diagnostics-y-by sli --first-n-reward-diagnostics-color-by none --best-worst-sli --best-worst-trn 1 --sli-use-training-mean --skip-first-sync-buckets 1 --sli-select-skip-first-sync-buckets 1 --first-n-reward-diagnostics-csv exports/sliT1SB2SB5_vs_rwdsFirst10_intact_ctrlKir_agaroseLgc_2026-04-22.csv --first-n-reward-diagnostics-plot-out exports/sliT1SB2SB5_vs_rwdsFirst10_intact_ctrlKir_agaroseLgc_2026-04-22.png",
            "first_n_reward_diagnostics_xlabel": None,
            "first_n_reward_diagnostics_ylabel": None,
            "output_path": "exports/sliT1SB2SB5_vs_rwdsFirst10_intact_ctrlKir_agaroseLgc_2026-04-22.png"
        }
    ],
}

panel_23


### Panel 23 Data Export Commands


In [ ]:
RUN_PANEL_23_EXPORTS = False
run_stage("Panel 23: export bundles", panel_23["export_commands"], run=RUN_PANEL_23_EXPORTS)


### Panel 23 Plotting Commands


In [ ]:
RUN_PANEL_23_PLOTS = False
run_stage("Panel 23: generate plot", panel_23["plot_commands"], run=RUN_PANEL_23_PLOTS)


## Template For Additional Panels

Copy this structure when you add a new panel:

1. Add a markdown cell with the panel title and biological description.
2. Copy the dictionary template below and rename it to the next available panel key, such as `panel_9`, `panel_10`, and so on.
3. Fill in the export commands for the new panel.
4. Fill in the plotting commands that consume the exported bundles.
5. Add one execution cell for exports and one for plotting.

In [ ]:
PANEL_TEMPLATE = {
    "title": "Panel N - Short title",
    "description": "One or two sentences describing the intended figure panel.",
    "export_commands": [
        {
            "label": "group or condition label",
            "command": "python analyze.py ... --export-... exports/example_bundle.npz ...",
        },
    ],
    "plot_commands": [
        {
            "label": "plot description",
            "command": "python -m scripts.some_plotter --bundles ... --out exports/example_plot.png",
            "xlabel": None,
            "ylabel": None,
            "rename_outputs": [
                {
                    "from": "exports/default_plot.png",
                    "to": "exports/example_plot.png",
                },
            ],
            "output_path": "exports/example_plot.png",
        },
    ],
}

PANEL_TEMPLATE